# Agent 1 — Module 3: Exact Notebook Migration

## Objective

This notebook migrates the current Module 3 implementation into Jupyter without changing the existing topic-extraction logic.

The following production components are copied into executable notebook cells exactly as they currently exist:

1. Topic schemas
2. Official AQA Computer Science concept catalogue
3. Official-topic candidate extractor
4. CS relevance filter
5. Unmapped-CS detector
6. Topic merger
7. Module 3 pipeline

The notebook then executes the same existing project classes on real `chunks.json` files generated by Module 2.

No topic name, expected topic list, confidence score, classification, candidate score, AQA reference, or chunk result is manually assigned.

The real flow is:

```text
Real Module 2 chunks
    ↓
Official AQA candidate extraction
    ↓
Keyword + MiniLM semantic evidence
    ↓
Salience calculation
    ↓
Candidate relevance filtering
    ↓
Continuation/no-new-topic handling
    ↓
Unmapped CS detection
    ↓
LLM-fallback decision flags
    ↓
Repeated-topic merging and ranking
    ↓
Real Module 3 result
```

# 0. Environment Setup

Run the notebook from the `Agent_1` project root.

The setup installs the current project requirements and enables imports from the existing application package.

In [ ]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in [
        start,
        *start.parents,
        start / "Agent_1",
    ]:
        if (
            (candidate / "app").is_dir()
            and (candidate / "requirements.txt").is_file()
        ):
            return candidate.resolve()

    raise RuntimeError(
        "Agent_1 project root was not found. "
        "Place this notebook inside the Agent_1 folder."
    )


PROJECT_ROOT = find_project_root(
    Path.cwd().resolve()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        str(PROJECT_ROOT / "requirements.txt"),
    ]
)

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "numpy",
        "pandas",
        "sentence-transformers",
    ]
)

print(f"Python executable: {sys.executable}")
print(f"Project root: {PROJECT_ROOT}")

# 1. Exact Existing Module 3 Implementation

Each implementation cell below is copied directly from the current Agent 1 project.

These cells document the complete implementation being migrated. The execution section later imports the same project classes so the notebook and application use one source of truth.

## Topic schemas

**Exact source:** `app/schemas/topic.py`

The following cell contains the current source code without notebook-specific changes.

In [ ]:
from __future__ import annotations

from typing import Literal

from pydantic import BaseModel, Field


ExtractionMethod = Literal[
    "keyword",
    "embedding",
    "keyword_embedding",
]

ChunkClassification = Literal[
    "official_aqa_topic",
    "mixed_official_and_unmapped",
    "cs_related_unmapped",
    "continuation_no_new_topic",
    "no_topic",
]

UnmappedDetectionMethod = Literal[
    "lexical",
    "semantic",
    "lexical_semantic",
]

TopicRole = Literal[
    "primary",
    "supporting",
]


class RawTopicCandidate(BaseModel):
    """
    Official AQA topic candidate before the final relevance decision.
    """

    concept_id: str = Field(min_length=1)
    topic: str = Field(min_length=1)
    domain: str = Field(min_length=1)

    official_reference: str = Field(min_length=1)
    chapter_reference: str = Field(min_length=1)
    official_title: str = Field(min_length=1)
    paper: str = Field(min_length=1)
    source_pages: list[int] = Field(default_factory=list)

    confidence: float = Field(ge=0.0, le=1.0)
    keyword_score: float = Field(ge=0.0, le=1.0)
    semantic_score: float = Field(ge=-1.0, le=1.0)
    salience_score: float = Field(ge=0.0, le=1.0)

    extraction_method: ExtractionMethod

    matched_aliases: list[str] = Field(default_factory=list)
    total_alias_hits: int = Field(default=0, ge=0)
    evidence_sentence_count: int = Field(default=0, ge=0)
    single_word_alias_only: bool = False

    evidence: list[str] = Field(default_factory=list)
    parent_concept_id: str | None = None


class TopicCandidate(RawTopicCandidate):
    """
    Candidate after the official-topic relevance filter.
    """

    cs_relevance_score: float = Field(ge=0.0, le=1.0)
    cs_relevant: bool


class UnmappedCSSignal(BaseModel):
    """
    Evidence that a chunk contains a Computer Science concept which was not
    confidently covered by an official AQA catalogue candidate.

    rough_topic is intentionally not an AQA mapping. It is a generic label
    that can later be reviewed or refined by the fallback stage.
    """

    rough_topic: str = Field(min_length=1)
    domain: str = Field(min_length=1)
    score: float = Field(ge=-1.0, le=1.0)
    evidence: str = Field(min_length=1)
    matched_aliases: list[str] = Field(default_factory=list)
    detection_method: UnmappedDetectionMethod


class ChunkTopicResult(BaseModel):
    """
    Complete Module 3 output for one transcript chunk.
    """

    chunk_id: int = Field(ge=1)
    source_word_count: int = Field(default=0, ge=0)

    classification: ChunkClassification = "no_topic"

    # A continuation-only chunk does not create a new topic.
    is_cs_relevant: bool
    creates_new_topic: bool = False

    cs_relevance_score: float = Field(ge=0.0, le=1.0)

    topic_candidates: list[TopicCandidate] = Field(default_factory=list)
    rejected_candidates: list[TopicCandidate] = Field(default_factory=list)

    has_unmapped_cs_content: bool = False
    unmapped_cs_signals: list[UnmappedCSSignal] = Field(default_factory=list)

    continuation_of_chunk_id: int | None = Field(default=None, ge=1)

    requires_llm_fallback: bool = False
    notes: list[str] = Field(default_factory=list)


class MergedTopic(BaseModel):
    """
    One official AQA topic merged across one or more transcript chunks.

    confidence remains the extraction confidence. ranking_score is used only
    to order lesson topics more realistically by semantic strength, salience
    and coverage rather than by repeated keyword matches alone.
    """

    concept_id: str = Field(min_length=1)
    topic: str = Field(min_length=1)
    domain: str = Field(min_length=1)

    official_reference: str = Field(min_length=1)
    chapter_reference: str = Field(min_length=1)
    official_title: str = Field(min_length=1)
    paper: str = Field(min_length=1)
    source_pages: list[int] = Field(default_factory=list)

    confidence: float = Field(ge=0.0, le=1.0)
    ranking_score: float = Field(ge=0.0, le=1.0)
    topic_role: TopicRole

    source_chunk_ids: list[int] = Field(default_factory=list)
    support_span_count: int = Field(default=1, ge=1)

    mean_semantic_score: float = Field(ge=-1.0, le=1.0)
    mean_keyword_score: float = Field(ge=0.0, le=1.0)
    mean_salience_score: float = Field(ge=0.0, le=1.0)
    coverage_score: float = Field(ge=0.0, le=1.0)

    evidence: list[str] = Field(default_factory=list)
    supporting_candidate_count: int = Field(ge=1)


class Module3Result(BaseModel):
    """
    Complete output of Module 3.
    """

    chunk_results: list[ChunkTopicResult]
    merged_topics: list[MergedTopic]

    total_chunks: int = Field(ge=0)
    cs_relevant_chunks: int = Field(ge=0)
    non_cs_chunks: int = Field(ge=0)

    official_topic_chunks: int = Field(default=0, ge=0)
    mixed_official_unmapped_chunks: int = Field(default=0, ge=0)
    unmapped_cs_chunks: int = Field(default=0, ge=0)
    continuation_chunks: int = Field(default=0, ge=0)
    no_topic_chunks: int = Field(default=0, ge=0)

    llm_fallback_chunk_ids: list[int] = Field(default_factory=list)

    embedding_model: str
    candidate_keep_threshold: float = Field(ge=0.0, le=1.0)

## AQA CS concept catalogue

**Exact source:** `app/services/cs_concept_catalog.py`

The following cell contains the current source code without notebook-specific changes.

In [ ]:
from __future__ import annotations

from collections import defaultdict
import re
from dataclasses import dataclass
from typing import Literal


Paper = Literal["Paper 1", "Paper 2"]


@dataclass(frozen=True)
class FlexiblePattern:
    """
    Reusable flexible lexical pattern for natural classroom language.

    Exact aliases are ideal when the transcript contains stable technical
    wording. Patterns handle grammatical variation, passive voice and short
    inserted tokens without tying the extractor to one transcript.
    """

    label: str
    regex: str
    weight: float = 0.82


@dataclass(frozen=True)
class CSConcept:
    """
    One searchable concept derived from the official AQA GCSE
    Computer Science (8525) specification.

    Important:
    - official_reference, chapter_title and official_title come from
      the AQA specification structure.
    - label and description are concise retrieval-friendly summaries.
    - aliases are transcript-friendly matching terms. They are not
      presented as official AQA wording.
    - Several searchable concepts may share one official reference
      when AQA groups multiple assessable ideas under one section.
    """

    concept_id: str

    # Official AQA hierarchy
    official_reference: str
    chapter_reference: str
    chapter_title: str
    official_title: str

    # Retrieval-friendly fields used by Module 3
    label: str
    domain: str
    description: str
    aliases: tuple[str, ...]

    paper: Paper
    source_pages: tuple[int, ...]

    parent_concept_id: str | None = None

    # Phrases that contain an alias but represent a different concept.
    # This metadata is reusable for any catalogue entry and prevents a
    # shorter term from consuming a longer compound term.
    excluded_phrases: tuple[str, ...] = ()

    # Flexible regex patterns for concepts whose classroom wording can vary.
    # Patterns are evaluated against normalised sentence text.
    match_patterns: tuple[FlexiblePattern, ...] = ()

    @property
    def embedding_text(self) -> str:
        """
        Text that can be embedded for semantic topic retrieval.
        """

        aliases = ", ".join(self.aliases)

        return (
            f"AQA GCSE Computer Science {self.official_reference}. "
            f"Chapter: {self.chapter_title}. "
            f"Official topic: {self.official_title}. "
            f"Search concept: {self.label}. "
            f"{self.description} "
            f"Related transcript terms: {aliases}."
        )


def _concept(
    *,
    concept_id: str,
    official_reference: str,
    chapter_reference: str,
    chapter_title: str,
    official_title: str,
    label: str,
    description: str,
    aliases: tuple[str, ...],
    paper: Paper,
    source_pages: tuple[int, ...],
    parent_concept_id: str | None = None,
    excluded_phrases: tuple[str, ...] = (),
    match_patterns: tuple[FlexiblePattern, ...] = (),
) -> CSConcept:
    """
    Small constructor that keeps domain aligned with the official
    AQA chapter title.
    """

    return CSConcept(
        concept_id=concept_id,
        official_reference=official_reference,
        chapter_reference=chapter_reference,
        chapter_title=chapter_title,
        official_title=official_title,
        label=label,
        domain=chapter_title,
        description=description,
        aliases=aliases,
        paper=paper,
        source_pages=source_pages,
        parent_concept_id=parent_concept_id,
        excluded_phrases=excluded_phrases,
        match_patterns=match_patterns,
    )


# =============================================================================
# OFFICIAL AQA CHAPTERS
# =============================================================================

AQA_CHAPTERS: dict[str, str] = {
    "3.1": "Fundamentals of algorithms",
    "3.2": "Programming",
    "3.3": "Fundamentals of data representation",
    "3.4": "Computer systems",
    "3.5": "Fundamentals of computer networks",
    "3.6": "Cyber security",
    "3.7": (
        "Relational databases and structured query language (SQL)"
    ),
    "3.8": (
        "Ethical, legal and environmental impacts of digital "
        "technology on wider society, including issues of privacy"
    ),
}


# =============================================================================
# SEARCHABLE OFFICIAL SYLLABUS CATALOGUE
# =============================================================================

CS_CONCEPTS: tuple[CSConcept, ...] = (
    # =========================================================================
    # 3.1 FUNDAMENTALS OF ALGORITHMS — PAPER 1
    # =========================================================================
    _concept(
        concept_id="aqa_3_1_1_algorithm",
        official_reference="3.1.1",
        chapter_reference="3.1",
        chapter_title=AQA_CHAPTERS["3.1"],
        official_title="Representing algorithms",
        label="Algorithms",
        description=(
            "Understand an algorithm as a sequence of steps and distinguish "
            "an algorithm from its implementation as a computer program."
        ),
        aliases=(
            "algorithm",
            "sequence of steps",
            "solve a task",
            "computer program implementation",
        ),
        paper="Paper 1",
        source_pages=(10,),
    ),
    _concept(
        concept_id="aqa_3_1_1_decomposition",
        official_reference="3.1.1",
        chapter_reference="3.1",
        chapter_title=AQA_CHAPTERS["3.1"],
        official_title="Representing algorithms",
        label="Decomposition",
        description=(
            "Break a problem into smaller sub-problems that each perform "
            "an identifiable task."
        ),
        aliases=(
            "decomposition",
            "break the problem down",
            "split into sub problems",
            "smaller subproblems",
        ),
        paper="Paper 1",
        source_pages=(10,),
        parent_concept_id="aqa_3_1_1_algorithm",
    ),
    _concept(
        concept_id="aqa_3_1_1_abstraction",
        official_reference="3.1.1",
        chapter_reference="3.1",
        chapter_title=AQA_CHAPTERS["3.1"],
        official_title="Representing algorithms",
        label="Abstraction",
        description=(
            "Remove unnecessary detail from a problem so attention remains "
            "on the information required for a solution."
        ),
        aliases=(
            "abstraction",
            "remove unnecessary detail",
            "ignore irrelevant detail",
            "focus on important information",
        ),
        paper="Paper 1",
        source_pages=(10,),
        parent_concept_id="aqa_3_1_1_algorithm",
    ),
    _concept(
        concept_id="aqa_3_1_1_algorithm_representation",
        official_reference="3.1.1",
        chapter_reference="3.1",
        chapter_title=AQA_CHAPTERS["3.1"],
        official_title="Representing algorithms",
        label="Pseudocode, program code and flowcharts",
        description=(
            "Create and represent algorithms systematically using "
            "pseudocode, program code and flowcharts."
        ),
        aliases=(
            "pseudocode",
            "pseudo code",
            "flowchart",
            "program code",
            "represent the algorithm",
            "algorithm design",
        ),
        paper="Paper 1",
        source_pages=(10,),
        parent_concept_id="aqa_3_1_1_algorithm",
    ),
    _concept(
        concept_id="aqa_3_1_1_algorithm_purpose_trace",
        official_reference="3.1.1",
        chapter_reference="3.1",
        chapter_title=AQA_CHAPTERS["3.1"],
        official_title="Representing algorithms",
        label="Algorithm tracing and program execution",
        description=(
            "Identify inputs, processing and outputs, then trace program or "
            "algorithm execution using trace tables, visual inspection and "
            "step-by-step value tracking to determine behaviour or purpose."
        ),
        aliases=(
            "input processing output",
            "inputs processing outputs",
            "purpose of the algorithm",
            "trace table",
            "visual inspection",
            "dry run",
            "dry run the program",
            "trace the algorithm",
            "trace the code",
            "follow the code step by step",
            "follow program execution",
            "track variable values",
            "count statement executions",
            "number of times a statement executes",
            "how many times the loop runs",
            "how many times the loop is executed",
            "statement execution count",
        ),
        match_patterns=(
            FlexiblePattern(
                label="counting statement or loop executions",
                regex=(
                    r"\b(?:how many|number of)\s+times\b"
                    r".{0,60}\b(?:statement|instruction|line|loop)\b"
                    r".{0,60}\b(?:run|runs|ran|running|execute|"
                    r"executes|executed|done)\b"
                ),
                weight=0.84,
            ),
            FlexiblePattern(
                label="reported execution count",
                regex=(
                    r"\b(?:statement|instruction|line|loop)\b"
                    r".{0,50}\b(?:run|runs|ran|running|execute|"
                    r"executes|executed|done)\b"
                    r".{0,35}\b(?:once|twice|[a-z]+\s+times|"
                    r"\d+\s+times)\b"
                ),
                weight=0.82,
            ),
            FlexiblePattern(
                label="tracking changing program values",
                regex=(
                    r"\b(?:value|variable|index|counter)\b"
                    r".{0,45}\b(?:becomes|changes|changed|updated|"
                    r"increases|decreases)\b"
                ),
                weight=0.78,
            ),
            FlexiblePattern(
                label="reasoning about possible execution behaviour",
                regex=(
                    r"\b(?:possible|impossible)\b"
                    r".{0,60}\b(?:statement|instruction|loop|"
                    r"execution|run)\b"
                ),
                weight=0.78,
            ),
        ),
        paper="Paper 1",
        source_pages=(10,),
        parent_concept_id="aqa_3_1_1_algorithm",
    ),
    _concept(
        concept_id="aqa_3_1_2_efficiency",
        official_reference="3.1.2",
        chapter_reference="3.1",
        chapter_title=AQA_CHAPTERS["3.1"],
        official_title="Efficiency of algorithms",
        label="Time efficiency of algorithms",
        description=(
            "Compare algorithms that solve the same problem and explain why "
            "one may be more time-efficient than another."
        ),
        aliases=(
            "algorithm efficiency",
            "time efficiency",
            "more efficient algorithm",
            "compare efficiency",
            "faster algorithm",
            "same problem",
        ),
        paper="Paper 1",
        source_pages=(10,),
    ),
    _concept(
        concept_id="aqa_3_1_3_linear_search",
        official_reference="3.1.3",
        chapter_reference="3.1",
        chapter_title=AQA_CHAPTERS["3.1"],
        official_title="Searching algorithms",
        label="Linear search",
        description=(
            "Understand and explain the mechanics of the linear search "
            "algorithm."
        ),
        aliases=(
            "linear search",
            "sequential search",
            "check each item",
            "search one by one",
            "first item to last item",
        ),
        paper="Paper 1",
        source_pages=(11,),
    ),
    _concept(
        concept_id="aqa_3_1_3_binary_search",
        official_reference="3.1.3",
        chapter_reference="3.1",
        chapter_title=AQA_CHAPTERS["3.1"],
        official_title="Searching algorithms",
        label="Binary search",
        description=(
            "Understand and explain the mechanics of binary search on "
            "ordered data."
        ),
        aliases=(
            "binary search",
            "middle item",
            "middle value",
            "discard half",
            "search sorted data",
            "lower half",
            "upper half",
        ),
        paper="Paper 1",
        source_pages=(11,),
    ),
    _concept(
        concept_id="aqa_3_1_3_search_comparison",
        official_reference="3.1.3",
        chapter_reference="3.1",
        chapter_title=AQA_CHAPTERS["3.1"],
        official_title="Searching algorithms",
        label="Comparing linear and binary search",
        description=(
            "Compare the advantages and disadvantages of linear and binary "
            "search algorithms."
        ),
        aliases=(
            "compare linear and binary search",
            "linear versus binary search",
            "advantages of binary search",
            "disadvantages of linear search",
            "search algorithm comparison",
        ),
        paper="Paper 1",
        source_pages=(11,),
    ),
    _concept(
        concept_id="aqa_3_1_4_sorting_algorithms",
        official_reference="3.1.4",
        chapter_reference="3.1",
        chapter_title=AQA_CHAPTERS["3.1"],
        official_title="Sorting algorithms",
        label="Sorting algorithms",
        description=(
            "Understand sorting as arranging data into an order and use "
            "the compare-and-swap idea when explaining sorting methods."
        ),
        aliases=(
            "sorting algorithm",
            "sorting algorithms",
            "sort the values",
            "arrange values in order",
            "compare and swap",
            "sorting by swapping",
        ),
        paper="Paper 1",
        source_pages=(11,),
    ),
    _concept(
        concept_id="aqa_3_1_4_merge_sort",
        official_reference="3.1.4",
        chapter_reference="3.1",
        chapter_title=AQA_CHAPTERS["3.1"],
        official_title="Sorting algorithms",
        label="Merge sort",
        description=(
            "Understand and explain the mechanics of the merge sort "
            "algorithm."
        ),
        aliases=(
            "merge sort",
            "split the list",
            "merge sorted lists",
            "divide and merge",
            "divide and conquer sort",
        ),
        paper="Paper 1",
        source_pages=(11,),
        parent_concept_id="aqa_3_1_4_sorting_algorithms",
    ),
    _concept(
        concept_id="aqa_3_1_4_bubble_sort",
        official_reference="3.1.4",
        chapter_reference="3.1",
        chapter_title=AQA_CHAPTERS["3.1"],
        official_title="Sorting algorithms",
        label="Bubble sort",
        description=(
            "Understand and explain the mechanics of the bubble sort "
            "algorithm."
        ),
        aliases=(
            "bubble sort",
            "compare adjacent values",
            "adjacent items",
            "swap adjacent",
            "sorting pass",
            "no swaps",
        ),
        paper="Paper 1",
        source_pages=(11,),
        parent_concept_id="aqa_3_1_4_sorting_algorithms",
    ),
    _concept(
        concept_id="aqa_3_1_4_sort_comparison",
        official_reference="3.1.4",
        chapter_reference="3.1",
        chapter_title=AQA_CHAPTERS["3.1"],
        official_title="Sorting algorithms",
        label="Comparing merge sort and bubble sort",
        description=(
            "Compare the advantages and disadvantages of merge sort and "
            "bubble sort."
        ),
        aliases=(
            "compare merge and bubble sort",
            "merge sort versus bubble sort",
            "sorting algorithm comparison",
            "advantages of merge sort",
            "disadvantages of bubble sort",
        ),
        paper="Paper 1",
        source_pages=(11,),
        parent_concept_id="aqa_3_1_4_sorting_algorithms",
    ),

    # =========================================================================
    # 3.2 PROGRAMMING — PAPER 1
    # =========================================================================
    _concept(
        concept_id="aqa_3_2_1_data_types",
        official_reference="3.2.1",
        chapter_reference="3.2",
        chapter_title=AQA_CHAPTERS["3.2"],
        official_title="Data types",
        label="Data types",
        description=(
            "Understand and use integer, real, Boolean, character and "
            "string data types appropriately."
        ),
        aliases=(
            "data type",
            "integer",
            "real number",
            "float",
            "boolean",
            "character",
            "string",
        ),
        paper="Paper 1",
        source_pages=(11, 12),
    ),
    _concept(
        concept_id="aqa_3_2_2_variables_constants_assignment",
        official_reference="3.2.2",
        chapter_reference="3.2",
        chapter_title=AQA_CHAPTERS["3.2"],
        official_title="Programming concepts",
        label="Variables, constants and assignment",
        description=(
            "Use variable declarations, constant declarations and "
            "assignment statements, and understand why named variables and "
            "constants are used."
        ),
        aliases=(
            "variable declaration",
            "constant declaration",
            "assignment statement",
            "assign a value",
            "named constant",
            "variable value",
        ),
        paper="Paper 1",
        source_pages=(12,),
    ),
    _concept(
        concept_id="aqa_3_2_2_iteration",
        official_reference="3.2.2",
        chapter_reference="3.2",
        chapter_title=AQA_CHAPTERS["3.2"],
        official_title="Programming concepts",
        label="Iteration",
        description=(
            "Use definite count-controlled and indefinite "
            "condition-controlled iteration, including conditions at the "
            "start or end of a loop."
        ),
        aliases=(
            "iteration",
            "repetition",
            "for loop",
            "while loop",
            "repeat until",
            "do while",
            "count controlled loop",
            "condition controlled loop",
            "loop condition",
        ),
        paper="Paper 1",
        source_pages=(12, 13),
    ),
    _concept(
        concept_id="aqa_3_2_2_selection",
        official_reference="3.2.2",
        chapter_reference="3.2",
        chapter_title=AQA_CHAPTERS["3.2"],
        official_title="Programming concepts",
        label="Selection",
        description=(
            "Use selection statements to choose which instructions execute."
        ),
        aliases=(
            "selection",
            "if statement",
            "else statement",
            "conditional statement",
            "if condition",
            "choice",
        ),
        paper="Paper 1",
        source_pages=(12, 13),
    ),
    _concept(
        concept_id="aqa_3_2_2_subroutine_statement",
        official_reference="3.2.2",
        chapter_reference="3.2",
        chapter_title=AQA_CHAPTERS["3.2"],
        official_title="Programming concepts",
        label="Subroutine statements",
        description=(
            "Use procedures or functions as statement types within "
            "programs."
        ),
        aliases=(
            "subroutine",
            "procedure",
            "function",
            "method call",
            "call the function",
        ),
        paper="Paper 1",
        source_pages=(12,),
        parent_concept_id="aqa_3_2_10_subroutines",
    ),
    _concept(
        concept_id="aqa_3_2_2_nested_structures",
        official_reference="3.2.2",
        chapter_reference="3.2",
        chapter_title=AQA_CHAPTERS["3.2"],
        official_title="Programming concepts",
        label="Nested selection and iteration",
        description=(
            "Use selection inside selection and iteration inside iteration "
            "or other control structures."
        ),
        aliases=(
            "nested selection",
            "nested if",
            "nested iteration",
            "nested loop",
            "loop inside a loop",
            "if inside if",
        ),
        paper="Paper 1",
        source_pages=(13,),
    ),
    _concept(
        concept_id="aqa_3_2_2_identifiers",
        official_reference="3.2.2",
        chapter_reference="3.2",
        chapter_title=AQA_CHAPTERS["3.2"],
        official_title="Programming concepts",
        label="Meaningful identifier names",
        description=(
            "Use meaningful names for variables, constants and "
            "subroutines, and explain why they improve programs."
        ),
        aliases=(
            "identifier name",
            "meaningful variable name",
            "meaningful identifier",
            "constant name",
            "subroutine name",
            "descriptive name",
        ),
        paper="Paper 1",
        source_pages=(13,),
    ),
    _concept(
        concept_id="aqa_3_2_3_arithmetic_operations",
        official_reference="3.2.3",
        chapter_reference="3.2",
        chapter_title=AQA_CHAPTERS["3.2"],
        official_title="Arithmetic operations in a programming language",
        label="Arithmetic operations",
        description=(
            "Use addition, subtraction, multiplication, real division, "
            "integer division and remainders."
        ),
        aliases=(
            "arithmetic operation",
            "addition",
            "subtraction",
            "multiplication",
            "real division",
            "integer division",
            "div operator",
            "mod operator",
            "remainder",
            "modulo",
        ),
        paper="Paper 1",
        source_pages=(14,),
    ),
    _concept(
        concept_id="aqa_3_2_4_relational_operations",
        official_reference="3.2.4",
        chapter_reference="3.2",
        chapter_title=AQA_CHAPTERS["3.2"],
        official_title="Relational operations in a programming language",
        label="Relational operators",
        description=(
            "Use and interpret equality, inequality and ordered comparison "
            "operators in algorithms and programs."
        ),
        aliases=(
            "relational operator",
            "equal to",
            "not equal to",
            "less than",
            "greater than",
            "less than or equal",
            "greater than or equal",
            "comparison operator",
        ),
        paper="Paper 1",
        source_pages=(14,),
    ),
    _concept(
        concept_id="aqa_3_2_5_boolean_operations",
        official_reference="3.2.5",
        chapter_reference="3.2",
        chapter_title=AQA_CHAPTERS["3.2"],
        official_title="Boolean operations in a programming language",
        label="Boolean operations in programming",
        description=(
            "Use NOT, AND and OR, including combinations of these "
            "operations in iteration and selection conditions."
        ),
        aliases=(
            "boolean operation",
            "and operator",
            "or operator",
            "not operator",
            "boolean condition",
            "both conditions",
            "at least one condition",
        ),
        paper="Paper 1",
        source_pages=(14,),
    ),
    _concept(
        concept_id="aqa_3_2_6_data_structures",
        official_reference="3.2.6",
        chapter_reference="3.2",
        chapter_title=AQA_CHAPTERS["3.2"],
        official_title="Data structures",
        label="Data structures",
        description=(
            "Understand data structures and use suitable structures when "
            "designing solutions."
        ),
        aliases=(
            "data structure",
            "store multiple values",
            "organise data",
            "structured data",
        ),
        paper="Paper 1",
        source_pages=(14, 15),
    ),
    _concept(
        concept_id="aqa_3_2_6_arrays",
        official_reference="3.2.6",
        chapter_reference="3.2",
        chapter_title=AQA_CHAPTERS["3.2"],
        official_title="Data structures",
        label="One- and two-dimensional arrays",
        description=(
            "Use one-dimensional and two-dimensional arrays, or equivalent "
            "structures, to solve simple problems."
        ),
        aliases=(
            "array",
            "arrays",
            "one dimensional array",
            "1d array",
            "two dimensional array",
            "2d array",
            "array index",
            "array length",
            "array traversal",
            "traverse the array",
        ),
        excluded_phrases=(
            "array list",
            "array lists",
            "arraylist",
            "arraylists",
            "dynamic array",
            "resizable array",
        ),
        paper="Paper 1",
        source_pages=(15,),
        parent_concept_id="aqa_3_2_6_data_structures",
    ),
    _concept(
        concept_id="aqa_3_2_6_records",
        official_reference="3.2.6",
        chapter_reference="3.2",
        chapter_title=AQA_CHAPTERS["3.2"],
        official_title="Data structures",
        label="Records",
        description=(
            "Use records, or equivalent structures, in solutions to "
            "simple problems."
        ),
        aliases=(
            "record",
            "record definition",
            "structured record",
            "fields in a record",
            "record data structure",
        ),
        paper="Paper 1",
        source_pages=(15,),
        parent_concept_id="aqa_3_2_6_data_structures",
    ),
    _concept(
        concept_id="aqa_3_2_7_input_output",
        official_reference="3.2.7",
        chapter_reference="3.2",
        chapter_title=AQA_CHAPTERS["3.2"],
        official_title="Input/output",
        label="Program input and output",
        description=(
            "Obtain user input from the keyboard and output program data "
            "or information to the display."
        ),
        aliases=(
            "user input",
            "keyboard input",
            "input statement",
            "output statement",
            "display output",
            "print output",
        ),
        paper="Paper 1",
        source_pages=(15,),
    ),
    _concept(
        concept_id="aqa_3_2_8_string_handling",
        official_reference="3.2.8",
        chapter_reference="3.2",
        chapter_title=AQA_CHAPTERS["3.2"],
        official_title="String handling operations in a programming language",
        label="String handling",
        description=(
            "Use length, position, substring, concatenation, character-code "
            "conversion and string-number conversion operations."
        ),
        aliases=(
            "string handling",
            "string length",
            "substring",
            "concatenation",
            "character code",
            "ascii code conversion",
            "string to integer",
            "integer to string",
            "string conversion",
        ),
        paper="Paper 1",
        source_pages=(15,),
    ),
    _concept(
        concept_id="aqa_3_2_9_random_numbers",
        official_reference="3.2.9",
        chapter_reference="3.2",
        chapter_title=AQA_CHAPTERS["3.2"],
        official_title="Random number generation in a programming language",
        label="Random number generation",
        description=(
            "Use random number generation within computer programs."
        ),
        aliases=(
            "random number",
            "random number generation",
            "generate a random value",
            "random integer",
            "random function",
        ),
        paper="Paper 1",
        source_pages=(15,),
    ),
    _concept(
        concept_id="aqa_3_2_10_subroutines",
        official_reference="3.2.10",
        chapter_reference="3.2",
        chapter_title=AQA_CHAPTERS["3.2"],
        official_title=(
            "Structured programming and subroutines "
            "(procedures and functions)"
        ),
        label="Subroutines, procedures and functions",
        description=(
            "Understand named reusable blocks of code and explain the "
            "advantages of using subroutines."
        ),
        aliases=(
            "subroutine",
            "procedure",
            "function",
            "named block of code",
            "reusable code",
            "call the subroutine",
            "advantages of subroutines",
        ),
        paper="Paper 1",
        source_pages=(16,),
    ),
    _concept(
        concept_id="aqa_3_2_10_parameters_returns",
        official_reference="3.2.10",
        chapter_reference="3.2",
        chapter_title=AQA_CHAPTERS["3.2"],
        official_title=(
            "Structured programming and subroutines "
            "(procedures and functions)"
        ),
        label="Parameters and return values",
        description=(
            "Pass data into subroutines using parameters and pass data out "
            "using return values."
        ),
        aliases=(
            "parameter",
            "parameters",
            "argument",
            "arguments",
            "pass data to a function",
            "return value",
            "calling routine",
        ),
        paper="Paper 1",
        source_pages=(16,),
        parent_concept_id="aqa_3_2_10_subroutines",
    ),
    _concept(
        concept_id="aqa_3_2_10_local_variables",
        official_reference="3.2.10",
        chapter_reference="3.2",
        chapter_title=AQA_CHAPTERS["3.2"],
        official_title=(
            "Structured programming and subroutines "
            "(procedures and functions)"
        ),
        label="Local variables",
        description=(
            "Use local variables, understand their lifetime and scope, and "
            "explain why local variables are good practice."
        ),
        aliases=(
            "local variable",
            "variable scope",
            "only accessible in the function",
            "only exists during the subroutine",
            "local scope",
        ),
        paper="Paper 1",
        source_pages=(16,),
        parent_concept_id="aqa_3_2_10_subroutines",
    ),
    _concept(
        concept_id="aqa_3_2_10_structured_programming",
        official_reference="3.2.10",
        chapter_reference="3.2",
        chapter_title=AQA_CHAPTERS["3.2"],
        official_title=(
            "Structured programming and subroutines "
            "(procedures and functions)"
        ),
        label="Structured and modular programming",
        description=(
            "Describe structured programming with modular components, "
            "well-documented interfaces, local variables, parameters and "
            "return values."
        ),
        aliases=(
            "structured programming",
            "modular programming",
            "modularised program",
            "program modules",
            "documented interface",
            "advantages of structured programming",
        ),
        paper="Paper 1",
        source_pages=(16,),
    ),
    _concept(
        concept_id="aqa_3_2_11_validation",
        official_reference="3.2.11",
        chapter_reference="3.2",
        chapter_title=AQA_CHAPTERS["3.2"],
        official_title="Robust and secure programming",
        label="Data validation routines",
        description=(
            "Write validation routines that check input length, presence "
            "and whether values lie within an allowed range."
        ),
        aliases=(
            "data validation",
            "input validation",
            "range check",
            "length check",
            "presence check",
            "empty string check",
            "valid input",
        ),
        paper="Paper 1",
        source_pages=(17,),
    ),
    _concept(
        concept_id="aqa_3_2_11_authentication",
        official_reference="3.2.11",
        chapter_reference="3.2",
        chapter_title=AQA_CHAPTERS["3.2"],
        official_title="Robust and secure programming",
        label="Simple authentication routines",
        description=(
            "Write simple username-and-password authentication routines."
        ),
        aliases=(
            "authentication routine",
            "username and password",
            "login routine",
            "check username",
            "check password",
            "authenticate the user",
        ),
        paper="Paper 1",
        source_pages=(17,),
    ),
    _concept(
        concept_id="aqa_3_2_11_testing_test_data",
        official_reference="3.2.11",
        chapter_reference="3.2",
        chapter_title=AQA_CHAPTERS["3.2"],
        official_title="Robust and secure programming",
        label="Program testing and test data",
        description=(
            "Understand testing and select, justify and use normal, "
            "boundary and erroneous test data."
        ),
        aliases=(
            "program testing",
            "test data",
            "normal data",
            "typical data",
            "boundary data",
            "extreme data",
            "erroneous data",
            "invalid data",
            "test case",
        ),
        paper="Paper 1",
        source_pages=(17,),
    ),
    _concept(
        concept_id="aqa_3_2_11_errors_debugging",
        official_reference="3.2.11",
        chapter_reference="3.2",
        chapter_title=AQA_CHAPTERS["3.2"],
        official_title="Robust and secure programming",
        label="Syntax errors, logic errors and debugging",
        description=(
            "Correct errors and identify or categorise syntax and logic "
            "errors in algorithms and programs."
        ),
        aliases=(
            "syntax error",
            "logic error",
            "debugging",
            "debug the program",
            "correct the error",
            "find the error",
            "categorise the error",
        ),
        paper="Paper 1",
        source_pages=(17,),
    ),

    # =========================================================================
    # 3.3 FUNDAMENTALS OF DATA REPRESENTATION — PAPER 2
    # =========================================================================
    _concept(
        concept_id="aqa_3_3_1_number_bases",
        official_reference="3.3.1",
        chapter_reference="3.3",
        chapter_title=AQA_CHAPTERS["3.3"],
        official_title="Number bases",
        label="Decimal, binary and hexadecimal number bases",
        description=(
            "Understand decimal base 10, binary base 2 and hexadecimal "
            "base 16."
        ),
        aliases=(
            "number base",
            "decimal",
            "denary",
            "base ten",
            "binary",
            "base two",
            "hexadecimal",
            "base sixteen",
        ),
        paper="Paper 2",
        source_pages=(18,),
    ),
    _concept(
        concept_id="aqa_3_3_1_binary_representation",
        official_reference="3.3.1",
        chapter_reference="3.3",
        chapter_title=AQA_CHAPTERS["3.3"],
        official_title="Number bases",
        label="Binary representation of data and instructions",
        description=(
            "Understand that computers use binary bit patterns to "
            "represent data and instructions."
        ),
        aliases=(
            "computers use binary",
            "binary representation",
            "bit pattern",
            "represent data in binary",
            "binary instructions",
        ),
        paper="Paper 2",
        source_pages=(18,),
        parent_concept_id="aqa_3_3_1_number_bases",
    ),
    _concept(
        concept_id="aqa_3_3_1_hexadecimal_use",
        official_reference="3.3.1",
        chapter_reference="3.3",
        chapter_title=AQA_CHAPTERS["3.3"],
        official_title="Number bases",
        label="Why hexadecimal is used",
        description=(
            "Explain why hexadecimal is useful in computer science."
        ),
        aliases=(
            "why hexadecimal is used",
            "hex is shorter",
            "compact binary representation",
            "easier to read than binary",
            "hexadecimal in computer science",
        ),
        paper="Paper 2",
        source_pages=(18,),
        parent_concept_id="aqa_3_3_1_number_bases",
    ),
    _concept(
        concept_id="aqa_3_3_2_base_conversion",
        official_reference="3.3.2",
        chapter_reference="3.3",
        chapter_title=AQA_CHAPTERS["3.3"],
        official_title="Converting between number bases",
        label="Converting between binary, decimal and hexadecimal",
        description=(
            "Convert whole-number values in both directions between binary, "
            "decimal and hexadecimal, within the specified range."
        ),
        aliases=(
            "binary to decimal",
            "binary to denary",
            "decimal to binary",
            "denary to binary",
            "binary to hexadecimal",
            "hexadecimal to binary",
            "decimal to hexadecimal",
            "hexadecimal to decimal",
            "convert number bases",
        ),
        paper="Paper 2",
        source_pages=(18,),
    ),
    _concept(
        concept_id="aqa_3_3_3_bits_bytes",
        official_reference="3.3.3",
        chapter_reference="3.3",
        chapter_title=AQA_CHAPTERS["3.3"],
        official_title="Units of information",
        label="Bits and bytes",
        description=(
            "Know that a bit is a binary digit and a byte is a group of "
            "eight bits."
        ),
        aliases=(
            "byte",
            "eight bits",
            "8 bits",
            "binary digit",
            "unit of information",
        ),
        paper="Paper 2",
        source_pages=(18,),
    ),
    _concept(
        concept_id="aqa_3_3_3_storage_units",
        official_reference="3.3.3",
        chapter_reference="3.3",
        chapter_title=AQA_CHAPTERS["3.3"],
        official_title="Units of information",
        label="Decimal storage units",
        description=(
            "Know and compare kilo, mega, giga and tera quantities using "
            "decimal powers of ten."
        ),
        aliases=(
            "kilobyte",
            "megabyte",
            "gigabyte",
            "terabyte",
            "kb",
            "mb",
            "gb",
            "tb",
            "storage units",
            "decimal prefixes",
        ),
        paper="Paper 2",
        source_pages=(19,),
    ),
    _concept(
        concept_id="aqa_3_3_4_binary_addition",
        official_reference="3.3.4",
        chapter_reference="3.3",
        chapter_title=AQA_CHAPTERS["3.3"],
        official_title="Binary arithmetic",
        label="Binary addition",
        description=(
            "Add together up to three binary numbers within the stated "
            "eight-bit limits."
        ),
        aliases=(
            "binary addition",
            "add binary numbers",
            "binary sum",
            "carry in binary",
            "add bits",
        ),
        paper="Paper 2",
        source_pages=(19,),
    ),
    _concept(
        concept_id="aqa_3_3_4_binary_shifts",
        official_reference="3.3.4",
        chapter_reference="3.3",
        chapter_title=AQA_CHAPTERS["3.3"],
        official_title="Binary arithmetic",
        label="Logical binary shifts",
        description=(
            "Apply logical binary shifts and explain their use for "
            "multiplication or division by powers of two."
        ),
        aliases=(
            "binary shift",
            "logical shift",
            "left shift",
            "right shift",
            "multiply by powers of two",
            "divide by powers of two",
        ),
        paper="Paper 2",
        source_pages=(19,),
    ),
    _concept(
        concept_id="aqa_3_3_5_character_encoding",
        official_reference="3.3.5",
        chapter_reference="3.3",
        chapter_title=AQA_CHAPTERS["3.3"],
        official_title="Character encoding",
        label="ASCII and Unicode character encoding",
        description=(
            "Understand character sets, 7-bit ASCII and Unicode, convert "
            "between characters and codes, and explain the advantages of "
            "Unicode."
        ),
        aliases=(
            "character encoding",
            "character set",
            "ascii",
            "7 bit ascii",
            "unicode",
            "character code",
            "code point",
            "unicode versus ascii",
        ),
        paper="Paper 2",
        source_pages=(19, 20),
    ),
    _concept(
        concept_id="aqa_3_3_6_bitmap_images",
        official_reference="3.3.6",
        chapter_reference="3.3",
        chapter_title=AQA_CHAPTERS["3.3"],
        official_title="Representing images",
        label="Bitmap images, pixels and colour depth",
        description=(
            "Explain how bitmap images use pixels, image dimensions and "
            "colour depth."
        ),
        aliases=(
            "bitmap image",
            "pixel",
            "picture element",
            "image size",
            "image resolution",
            "width by height",
            "colour depth",
            "color depth",
            "bits per pixel",
        ),
        paper="Paper 2",
        source_pages=(20,),
    ),
    _concept(
        concept_id="aqa_3_3_6_image_file_size",
        official_reference="3.3.6",
        chapter_reference="3.3",
        chapter_title=AQA_CHAPTERS["3.3"],
        official_title="Representing images",
        label="Bitmap file-size calculations",
        description=(
            "Calculate bitmap file size from width, height and colour "
            "depth, and explain how pixels and colour depth affect size."
        ),
        aliases=(
            "bitmap file size",
            "image file size",
            "width times height times colour depth",
            "w h d",
            "pixels affect file size",
            "colour depth affects file size",
        ),
        paper="Paper 2",
        source_pages=(20,),
        parent_concept_id="aqa_3_3_6_bitmap_images",
    ),
    _concept(
        concept_id="aqa_3_3_6_bitmap_binary_conversion",
        official_reference="3.3.6",
        chapter_reference="3.3",
        chapter_title=AQA_CHAPTERS["3.3"],
        official_title="Representing images",
        label="Converting between bitmap images and binary data",
        description=(
            "Convert simple binary patterns into bitmap images and simple "
            "bitmap images into binary data."
        ),
        aliases=(
            "binary bitmap",
            "bitmap to binary",
            "binary to bitmap",
            "draw pixels from binary",
            "convert image to binary",
        ),
        paper="Paper 2",
        source_pages=(21,),
        parent_concept_id="aqa_3_3_6_bitmap_images",
    ),
    _concept(
        concept_id="aqa_3_3_7_sound_sampling",
        official_reference="3.3.7",
        chapter_reference="3.3",
        chapter_title=AQA_CHAPTERS["3.3"],
        official_title="Representing sound",
        label="Digital sound sampling",
        description=(
            "Explain analogue-to-digital sound conversion using samples, "
            "sampling rate and sample resolution."
        ),
        aliases=(
            "digital sound",
            "analogue sound",
            "analog sound",
            "sound sampling",
            "sample rate",
            "sampling rate",
            "sample resolution",
            "sample depth",
            "amplitude sample",
        ),
        paper="Paper 2",
        source_pages=(21,),
    ),
    _concept(
        concept_id="aqa_3_3_7_sound_file_size",
        official_reference="3.3.7",
        chapter_reference="3.3",
        chapter_title=AQA_CHAPTERS["3.3"],
        official_title="Representing sound",
        label="Sound file-size calculations",
        description=(
            "Calculate sound file size from sampling rate, sample "
            "resolution and duration."
        ),
        aliases=(
            "sound file size",
            "sampling rate times resolution times seconds",
            "audio file size",
            "rate res secs",
            "calculate sound size",
        ),
        paper="Paper 2",
        source_pages=(21,),
        parent_concept_id="aqa_3_3_7_sound_sampling",
    ),
    _concept(
        concept_id="aqa_3_3_8_compression",
        official_reference="3.3.8",
        chapter_reference="3.3",
        chapter_title=AQA_CHAPTERS["3.3"],
        official_title="Data compression",
        label="Data compression",
        description=(
            "Explain data compression, why it is used and that different "
            "compression methods exist."
        ),
        aliases=(
            "data compression",
            "compress data",
            "reduce file size",
            "compressed file",
            "why compress data",
        ),
        paper="Paper 2",
        source_pages=(21,),
    ),
    _concept(
        concept_id="aqa_3_3_8_huffman",
        official_reference="3.3.8",
        chapter_reference="3.3",
        chapter_title=AQA_CHAPTERS["3.3"],
        official_title="Data compression",
        label="Huffman coding",
        description=(
            "Explain and interpret Huffman coding and Huffman trees, and "
            "calculate compressed and uncompressed bit totals."
        ),
        aliases=(
            "huffman coding",
            "huffman tree",
            "huffman code",
            "variable length code",
            "compressed bits",
            "ascii bits",
        ),
        paper="Paper 2",
        source_pages=(21, 22),
        parent_concept_id="aqa_3_3_8_compression",
    ),
    _concept(
        concept_id="aqa_3_3_8_rle",
        official_reference="3.3.8",
        chapter_reference="3.3",
        chapter_title=AQA_CHAPTERS["3.3"],
        official_title="Data compression",
        label="Run-length encoding",
        description=(
            "Explain run-length encoding and represent data using "
            "frequency/data pairs."
        ),
        aliases=(
            "run length encoding",
            "rle",
            "frequency data pairs",
            "repeat count and value",
            "compress repeated values",
        ),
        paper="Paper 2",
        source_pages=(22,),
        parent_concept_id="aqa_3_3_8_compression",
    ),

    # =========================================================================
    # 3.4 COMPUTER SYSTEMS — PAPER 2
    # =========================================================================
    _concept(
        concept_id="aqa_3_4_1_hardware_software",
        official_reference="3.4.1",
        chapter_reference="3.4",
        chapter_title=AQA_CHAPTERS["3.4"],
        official_title="Hardware and software",
        label="Hardware and software",
        description=(
            "Define hardware and software and understand the relationship "
            "between them."
        ),
        aliases=(
            "hardware",
            "software",
            "physical components",
            "computer programs",
            "hardware and software relationship",
        ),
        paper="Paper 2",
        source_pages=(22,),
    ),
    _concept(
        concept_id="aqa_3_4_2_truth_tables_logic_gates",
        official_reference="3.4.2",
        chapter_reference="3.4",
        chapter_title=AQA_CHAPTERS["3.4"],
        official_title="Boolean logic",
        label="Logic gates and truth tables",
        description=(
            "Construct and interpret truth tables for NOT, AND, OR and XOR "
            "gates and combinations of those gates."
        ),
        aliases=(
            "logic gate",
            "truth table",
            "and gate",
            "or gate",
            "not gate",
            "xor gate",
            "logic circuit truth table",
        ),
        paper="Paper 2",
        source_pages=(22,),
    ),
    _concept(
        concept_id="aqa_3_4_2_logic_circuits",
        official_reference="3.4.2",
        chapter_reference="3.4",
        chapter_title=AQA_CHAPTERS["3.4"],
        official_title="Boolean logic",
        label="Logic circuit diagrams",
        description=(
            "Create, modify and interpret simple logic circuit diagrams "
            "using NOT, AND, OR and XOR."
        ),
        aliases=(
            "logic circuit",
            "circuit diagram",
            "draw logic gates",
            "interpret the circuit",
            "modify logic circuit",
        ),
        paper="Paper 2",
        source_pages=(23,),
        parent_concept_id="aqa_3_4_2_truth_tables_logic_gates",
    ),
    _concept(
        concept_id="aqa_3_4_2_boolean_expressions",
        official_reference="3.4.2",
        chapter_reference="3.4",
        chapter_title=AQA_CHAPTERS["3.4"],
        official_title="Boolean logic",
        label="Boolean expressions and circuits",
        description=(
            "Create and interpret Boolean expressions and convert between "
            "simple expressions and logic circuits."
        ),
        aliases=(
            "boolean expression",
            "logic expression",
            "expression from circuit",
            "circuit from expression",
            "a and b or not c",
        ),
        paper="Paper 2",
        source_pages=(23,),
        parent_concept_id="aqa_3_4_2_truth_tables_logic_gates",
    ),
    _concept(
        concept_id="aqa_3_4_3_software_classification",
        official_reference="3.4.3",
        chapter_reference="3.4",
        chapter_title=AQA_CHAPTERS["3.4"],
        official_title="Software classification",
        label="System software and application software",
        description=(
            "Explain system software and application software and give "
            "examples of both."
        ),
        aliases=(
            "system software",
            "application software",
            "software classification",
            "end user software",
            "platform software",
        ),
        paper="Paper 2",
        source_pages=(23,),
    ),
    _concept(
        concept_id="aqa_3_4_3_operating_systems_utilities",
        official_reference="3.4.3",
        chapter_reference="3.4",
        chapter_title=AQA_CHAPTERS["3.4"],
        official_title="Software classification",
        label="Operating systems and utility programs",
        description=(
            "Explain why operating systems and utilities are needed and "
            "describe management of processor, memory, I/O devices, "
            "applications and security."
        ),
        aliases=(
            "operating system",
            "os",
            "utility program",
            "processor management",
            "memory management",
            "device management",
            "application management",
            "operating system security",
        ),
        paper="Paper 2",
        source_pages=(24,),
        parent_concept_id="aqa_3_4_3_software_classification",
    ),
    _concept(
        concept_id="aqa_3_4_4_language_levels",
        official_reference="3.4.4",
        chapter_reference="3.4",
        chapter_title=AQA_CHAPTERS["3.4"],
        official_title=(
            "Classification of programming languages and translators"
        ),
        label="High-level and low-level programming languages",
        description=(
            "Compare high-level and low-level languages and explain "
            "machine code and assembly language."
        ),
        aliases=(
            "high level language",
            "low level language",
            "machine code",
            "assembly language",
            "one to one with machine code",
            "processor instruction set",
        ),
        paper="Paper 2",
        source_pages=(24,),
    ),
    _concept(
        concept_id="aqa_3_4_4_translators",
        official_reference="3.4.4",
        chapter_reference="3.4",
        chapter_title=AQA_CHAPTERS["3.4"],
        official_title=(
            "Classification of programming languages and translators"
        ),
        label="Compilers, interpreters and assemblers",
        description=(
            "Explain the differences between compiler, interpreter and "
            "assembler translators and when each is appropriate."
        ),
        aliases=(
            "compiler",
            "interpreter",
            "assembler",
            "program translator",
            "translate to machine code",
            "compiled code",
            "interpreted code",
        ),
        paper="Paper 2",
        source_pages=(25,),
    ),
    _concept(
        concept_id="aqa_3_4_5_cpu_von_neumann",
        official_reference="3.4.5",
        chapter_reference="3.4",
        chapter_title=AQA_CHAPTERS["3.4"],
        official_title="Systems architecture",
        label="CPU components and Von Neumann architecture",
        description=(
            "Explain main memory and CPU components including the ALU, "
            "control unit, clock, registers and buses."
        ),
        aliases=(
            "von neumann architecture",
            "cpu",
            "central processing unit",
            "arithmetic logic unit",
            "alu",
            "control unit",
            "clock",
            "register",
            "bus",
            "main memory",
        ),
        paper="Paper 2",
        source_pages=(25,),
    ),
    _concept(
        concept_id="aqa_3_4_5_cpu_performance",
        official_reference="3.4.5",
        chapter_reference="3.4",
        chapter_title=AQA_CHAPTERS["3.4"],
        official_title="Systems architecture",
        label="CPU performance",
        description=(
            "Explain how clock speed, number of processor cores and cache "
            "size affect CPU performance."
        ),
        aliases=(
            "cpu performance",
            "clock speed",
            "processor cores",
            "number of cores",
            "cache size",
            "faster processor",
        ),
        paper="Paper 2",
        source_pages=(25,),
        parent_concept_id="aqa_3_4_5_cpu_von_neumann",
    ),
    _concept(
        concept_id="aqa_3_4_5_fetch_execute_cycle",
        official_reference="3.4.5",
        chapter_reference="3.4",
        chapter_title=AQA_CHAPTERS["3.4"],
        official_title="Systems architecture",
        label="Fetch-decode-execute cycle",
        description=(
            "Explain how the CPU fetches, decodes and executes instructions "
            "stored in main memory."
        ),
        aliases=(
            "fetch execute cycle",
            "fetch decode execute",
            "instruction cycle",
            "fetch instruction",
            "decode instruction",
            "execute instruction",
        ),
        paper="Paper 2",
        source_pages=(25,),
        parent_concept_id="aqa_3_4_5_cpu_von_neumann",
    ),
    _concept(
        concept_id="aqa_3_4_5_memory",
        official_reference="3.4.5",
        chapter_reference="3.4",
        chapter_title=AQA_CHAPTERS["3.4"],
        official_title="Systems architecture",
        label="RAM, ROM, cache and registers",
        description=(
            "Explain the uses of RAM, ROM, cache and registers and compare "
            "main memory, secondary storage, volatile and non-volatile "
            "memory."
        ),
        aliases=(
            "ram",
            "rom",
            "cache memory",
            "register memory",
            "volatile memory",
            "non volatile memory",
            "main memory",
            "secondary storage",
            "ram versus rom",
        ),
        paper="Paper 2",
        source_pages=(26,),
    ),
    _concept(
        concept_id="aqa_3_4_5_secondary_storage",
        official_reference="3.4.5",
        chapter_reference="3.4",
        chapter_title=AQA_CHAPTERS["3.4"],
        official_title="Systems architecture",
        label="Secondary storage",
        description=(
            "Explain why secondary storage is required and compare solid "
            "state, optical and magnetic storage."
        ),
        aliases=(
            "secondary storage",
            "solid state storage",
            "ssd",
            "optical storage",
            "magnetic storage",
            "hard disk",
            "storage advantages disadvantages",
        ),
        paper="Paper 2",
        source_pages=(26,),
    ),
    _concept(
        concept_id="aqa_3_4_5_cloud_storage",
        official_reference="3.4.5",
        chapter_reference="3.4",
        chapter_title=AQA_CHAPTERS["3.4"],
        official_title="Systems architecture",
        label="Cloud storage",
        description=(
            "Explain cloud storage and compare remote cloud storage with "
            "local storage."
        ),
        aliases=(
            "cloud storage",
            "remote storage",
            "local storage",
            "store data online",
            "cloud versus local",
        ),
        paper="Paper 2",
        source_pages=(26,),
    ),
    _concept(
        concept_id="aqa_3_4_5_embedded_systems",
        official_reference="3.4.5",
        chapter_reference="3.4",
        chapter_title=AQA_CHAPTERS["3.4"],
        official_title="Systems architecture",
        label="Embedded systems",
        description=(
            "Explain how embedded systems differ from non-embedded systems "
            "and give examples."
        ),
        aliases=(
            "embedded system",
            "embedded computer",
            "dedicated system",
            "non embedded system",
            "computer inside a device",
        ),
        paper="Paper 2",
        source_pages=(26,),
    ),

    # =========================================================================
    # 3.5 FUNDAMENTALS OF COMPUTER NETWORKS — PAPER 2
    # =========================================================================
    _concept(
        concept_id="aqa_3_5_network_fundamentals",
        official_reference="3.5",
        chapter_reference="3.5",
        chapter_title=AQA_CHAPTERS["3.5"],
        official_title="Fundamentals of computer networks",
        label="Computer networks",
        description=(
            "Define computer networks and discuss their advantages and "
            "disadvantages."
        ),
        aliases=(
            "computer network",
            "network definition",
            "connected computers",
            "share resources",
            "advantages of networks",
            "disadvantages of networks",
        ),
        paper="Paper 2",
        source_pages=(27,),
    ),
    _concept(
        concept_id="aqa_3_5_network_types",
        official_reference="3.5",
        chapter_reference="3.5",
        chapter_title=AQA_CHAPTERS["3.5"],
        official_title="Fundamentals of computer networks",
        label="PAN, LAN and WAN",
        description=(
            "Describe personal, local and wide area networks, including "
            "typical scale, ownership and examples."
        ),
        aliases=(
            "pan",
            "personal area network",
            "bluetooth network",
            "lan",
            "local area network",
            "wan",
            "wide area network",
            "internet is a wan",
            "network types",
        ),
        paper="Paper 2",
        source_pages=(27,),
        parent_concept_id="aqa_3_5_network_fundamentals",
    ),
    _concept(
        concept_id="aqa_3_5_wired_wireless",
        official_reference="3.5",
        chapter_reference="3.5",
        chapter_title=AQA_CHAPTERS["3.5"],
        official_title="Fundamentals of computer networks",
        label="Wired and wireless networks",
        description=(
            "Compare wired and wireless networks, including copper and "
            "fibre cabling and appropriate uses."
        ),
        aliases=(
            "wired network",
            "wireless network",
            "wifi",
            "wi fi",
            "copper cable",
            "fibre cable",
            "fiber cable",
            "wired versus wireless",
        ),
        paper="Paper 2",
        source_pages=(27,),
        parent_concept_id="aqa_3_5_network_fundamentals",
    ),
    _concept(
        concept_id="aqa_3_5_topologies",
        official_reference="3.5",
        chapter_reference="3.5",
        chapter_title=AQA_CHAPTERS["3.5"],
        official_title="Fundamentals of computer networks",
        label="Star and bus network topologies",
        description=(
            "Describe, draw, compare and select star and bus LAN "
            "topologies."
        ),
        aliases=(
            "network topology",
            "star topology",
            "bus topology",
            "lan topology",
            "topology diagram",
            "central switch",
            "backbone cable",
        ),
        paper="Paper 2",
        source_pages=(27,),
        parent_concept_id="aqa_3_5_network_fundamentals",
    ),
    _concept(
        concept_id="aqa_3_5_protocols",
        official_reference="3.5",
        chapter_reference="3.5",
        chapter_title=AQA_CHAPTERS["3.5"],
        official_title="Fundamentals of computer networks",
        label="Network protocols",
        description=(
            "Define protocols and explain the purpose and use of Ethernet, "
            "Wi-Fi, TCP, UDP, IP, HTTP, HTTPS, FTP, SMTP and IMAP."
        ),
        aliases=(
            "network protocol",
            "protocol",
            "ethernet",
            "wifi protocol",
            "tcp",
            "udp",
            "ip protocol",
            "http",
            "https",
            "ftp",
            "smtp",
            "imap",
            "email protocol",
        ),
        paper="Paper 2",
        source_pages=(27, 28),
        parent_concept_id="aqa_3_5_network_fundamentals",
    ),
    _concept(
        concept_id="aqa_3_5_network_security",
        official_reference="3.5",
        chapter_reference="3.5",
        chapter_title=AQA_CHAPTERS["3.5"],
        official_title="Fundamentals of computer networks",
        label="Network security methods",
        description=(
            "Explain network security using authentication, encryption, "
            "firewalls and MAC address filtering, including how controls "
            "work together."
        ),
        aliases=(
            "network security",
            "authentication",
            "encryption",
            "firewall",
            "mac address filtering",
            "block network traffic",
            "allow network traffic",
            "security layers",
        ),
        paper="Paper 2",
        source_pages=(28,),
        parent_concept_id="aqa_3_5_network_fundamentals",
    ),
    _concept(
        concept_id="aqa_3_5_tcp_ip_model",
        official_reference="3.5",
        chapter_reference="3.5",
        chapter_title=AQA_CHAPTERS["3.5"],
        official_title="Fundamentals of computer networks",
        label="Four-layer TCP/IP model",
        description=(
            "Describe the application, transport, internet and link layers "
            "and place common protocols at the correct layer."
        ),
        aliases=(
            "tcp ip model",
            "four layer model",
            "4 layer model",
            "application layer",
            "transport layer",
            "internet layer",
            "link layer",
            "network access layer",
        ),
        paper="Paper 2",
        source_pages=(29,),
        parent_concept_id="aqa_3_5_network_fundamentals",
    ),

    # =========================================================================
    # 3.6 CYBER SECURITY — PAPER 2
    # =========================================================================
    _concept(
        concept_id="aqa_3_6_1_cyber_security",
        official_reference="3.6.1",
        chapter_reference="3.6",
        chapter_title=AQA_CHAPTERS["3.6"],
        official_title="Fundamentals of cyber security",
        label="Cyber security fundamentals",
        description=(
            "Define cyber security and describe its purpose in protecting "
            "networks, computers, programs and data."
        ),
        aliases=(
            "cyber security",
            "cybersecurity",
            "protect networks",
            "protect computers",
            "protect data",
            "unauthorised access",
            "unauthorized access",
        ),
        paper="Paper 2",
        source_pages=(29,),
    ),
    _concept(
        concept_id="aqa_3_6_2_cyber_threats",
        official_reference="3.6.2",
        chapter_reference="3.6",
        chapter_title=AQA_CHAPTERS["3.6"],
        official_title="Cyber security threats",
        label="Cyber security threats",
        description=(
            "Explain social engineering, malware, pharming, weak or "
            "default passwords, misconfigured access rights, removable "
            "media and unpatched software as threats."
        ),
        aliases=(
            "cyber threat",
            "security threat",
            "pharming",
            "weak password",
            "default password",
            "misconfigured access rights",
            "removable media",
            "unpatched software",
            "outdated software",
        ),
        paper="Paper 2",
        source_pages=(30,),
        parent_concept_id="aqa_3_6_1_cyber_security",
    ),
    _concept(
        concept_id="aqa_3_6_2_penetration_testing",
        official_reference="3.6.2",
        chapter_reference="3.6",
        chapter_title=AQA_CHAPTERS["3.6"],
        official_title="Cyber security threats",
        label="Penetration testing",
        description=(
            "Explain penetration testing and distinguish testing that "
            "simulates an internal attack from testing that simulates an "
            "external attack."
        ),
        aliases=(
            "penetration testing",
            "pen test",
            "ethical hacking",
            "test system security",
            "internal attack",
            "external attack",
            "test without credentials",
        ),
        paper="Paper 2",
        source_pages=(30,),
        parent_concept_id="aqa_3_6_1_cyber_security",
    ),
    _concept(
        concept_id="aqa_3_6_2_1_social_engineering",
        official_reference="3.6.2.1",
        chapter_reference="3.6",
        chapter_title=AQA_CHAPTERS["3.6"],
        official_title="Social engineering",
        label="Social engineering, blagging, phishing and shouldering",
        description=(
            "Define social engineering, explain how it is prevented and "
            "describe blagging, phishing and shouldering."
        ),
        aliases=(
            "social engineering",
            "blagging",
            "pretexting",
            "phishing",
            "fake email",
            "fraudulent message",
            "shouldering",
            "shoulder surfing",
            "steal confidential information",
        ),
        paper="Paper 2",
        source_pages=(30,),
        parent_concept_id="aqa_3_6_2_cyber_threats",
    ),
    _concept(
        concept_id="aqa_3_6_2_2_malware",
        official_reference="3.6.2.2",
        chapter_reference="3.6",
        chapter_title=AQA_CHAPTERS["3.6"],
        official_title="Malicious code (malware)",
        label="Malware, viruses, Trojans and spyware",
        description=(
            "Define malware, explain protection against it and describe "
            "computer viruses, Trojans and spyware."
        ),
        aliases=(
            "malware",
            "malicious code",
            "computer virus",
            "virus",
            "trojan",
            "spyware",
            "hostile software",
            "intrusive software",
        ),
        paper="Paper 2",
        source_pages=(31,),
        parent_concept_id="aqa_3_6_2_cyber_threats",
    ),
    _concept(
        concept_id="aqa_3_6_3_security_measures",
        official_reference="3.6.3",
        chapter_reference="3.6",
        chapter_title=AQA_CHAPTERS["3.6"],
        official_title=(
            "Methods to detect and prevent cyber security threats"
        ),
        label="Cyber security prevention measures",
        description=(
            "Explain biometric measures, password systems, CAPTCHA, email "
            "identity confirmations and automatic software updates."
        ),
        aliases=(
            "biometric security",
            "fingerprint security",
            "password system",
            "captcha",
            "email confirmation",
            "confirm user identity",
            "automatic software update",
            "security update",
        ),
        paper="Paper 2",
        source_pages=(31,),
        parent_concept_id="aqa_3_6_1_cyber_security",
    ),

    # =========================================================================
    # 3.7 RELATIONAL DATABASES AND SQL — PAPER 2
    # =========================================================================
    _concept(
        concept_id="aqa_3_7_1_database_fundamentals",
        official_reference="3.7.1",
        chapter_reference="3.7",
        chapter_title=AQA_CHAPTERS["3.7"],
        official_title="Relational databases",
        label="Database and relational database concepts",
        description=(
            "Explain databases and relational databases."
        ),
        aliases=(
            "database",
            "relational database",
            "database concept",
            "related tables",
            "store structured data",
        ),
        paper="Paper 2",
        source_pages=(31,),
    ),
    _concept(
        concept_id="aqa_3_7_1_database_structure",
        official_reference="3.7.1",
        chapter_reference="3.7",
        chapter_title=AQA_CHAPTERS["3.7"],
        official_title="Relational databases",
        label="Tables, records, fields and data types",
        description=(
            "Understand tables, records, fields and data types in a "
            "relational database."
        ),
        aliases=(
            "database table",
            "record",
            "database field",
            "column",
            "row",
            "database data type",
        ),
        paper="Paper 2",
        source_pages=(32,),
        parent_concept_id="aqa_3_7_1_database_fundamentals",
    ),
    _concept(
        concept_id="aqa_3_7_1_keys",
        official_reference="3.7.1",
        chapter_reference="3.7",
        chapter_title=AQA_CHAPTERS["3.7"],
        official_title="Relational databases",
        label="Primary keys and foreign keys",
        description=(
            "Understand primary keys and foreign keys in relational "
            "databases."
        ),
        aliases=(
            "primary key",
            "foreign key",
            "unique identifier",
            "link tables",
            "related table key",
        ),
        paper="Paper 2",
        source_pages=(32,),
        parent_concept_id="aqa_3_7_1_database_fundamentals",
    ),
    _concept(
        concept_id="aqa_3_7_1_redundancy_inconsistency",
        official_reference="3.7.1",
        chapter_reference="3.7",
        chapter_title=AQA_CHAPTERS["3.7"],
        official_title="Relational databases",
        label="Data redundancy and inconsistency",
        description=(
            "Explain how relational databases help reduce data "
            "inconsistency and data redundancy."
        ),
        aliases=(
            "data redundancy",
            "data inconsistency",
            "duplicate data",
            "remove repeated data",
            "relational database benefits",
        ),
        paper="Paper 2",
        source_pages=(32,),
        parent_concept_id="aqa_3_7_1_database_fundamentals",
    ),
    _concept(
        concept_id="aqa_3_7_2_sql_select",
        official_reference="3.7.2",
        chapter_reference="3.7",
        chapter_title=AQA_CHAPTERS["3.7"],
        official_title="Structured query language (SQL)",
        label="SQL data retrieval",
        description=(
            "Retrieve relational database data using SELECT, FROM, WHERE "
            "and ORDER BY with ascending or descending order."
        ),
        aliases=(
            "sql select",
            "select from",
            "where clause",
            "order by",
            "ascending order",
            "descending order",
            "retrieve database data",
            "sql query",
        ),
        paper="Paper 2",
        source_pages=(32,),
    ),
    _concept(
        concept_id="aqa_3_7_2_sql_insert",
        official_reference="3.7.2",
        chapter_reference="3.7",
        chapter_title=AQA_CHAPTERS["3.7"],
        official_title="Structured query language (SQL)",
        label="SQL INSERT",
        description=(
            "Insert data into a relational database using INSERT INTO and "
            "VALUES."
        ),
        aliases=(
            "insert into",
            "sql insert",
            "values clause",
            "add database record",
            "insert a row",
        ),
        paper="Paper 2",
        source_pages=(32,),
    ),
    _concept(
        concept_id="aqa_3_7_2_sql_update_delete",
        official_reference="3.7.2",
        chapter_reference="3.7",
        chapter_title=AQA_CHAPTERS["3.7"],
        official_title="Structured query language (SQL)",
        label="SQL UPDATE and DELETE",
        description=(
            "Edit and delete relational database data using UPDATE, SET, "
            "DELETE FROM and WHERE."
        ),
        aliases=(
            "sql update",
            "update set",
            "delete from",
            "sql delete",
            "edit database data",
            "delete a record",
            "where condition",
        ),
        paper="Paper 2",
        source_pages=(32,),
    ),

    # =========================================================================
    # 3.8 ETHICAL, LEGAL AND ENVIRONMENTAL IMPACTS — PAPER 2
    # =========================================================================
    _concept(
        concept_id="aqa_3_8_impacts",
        official_reference="3.8",
        chapter_reference="3.8",
        chapter_title=AQA_CHAPTERS["3.8"],
        official_title=AQA_CHAPTERS["3.8"],
        label="Ethical, legal and environmental impacts of technology",
        description=(
            "Explain current ethical, legal and environmental impacts and "
            "risks of digital technology on society."
        ),
        aliases=(
            "ethical impact",
            "legal impact",
            "environmental impact",
            "digital technology on society",
            "social impact of technology",
            "technology risks",
        ),
        paper="Paper 2",
        source_pages=(33,),
    ),
    _concept(
        concept_id="aqa_3_8_privacy",
        official_reference="3.8",
        chapter_reference="3.8",
        chapter_title=AQA_CHAPTERS["3.8"],
        official_title=AQA_CHAPTERS["3.8"],
        label="Privacy and access to personal data",
        description=(
            "Consider privacy issues and competing arguments about access "
            "to private data by governments and security services."
        ),
        aliases=(
            "data privacy",
            "personal privacy",
            "private data",
            "government access to data",
            "security services",
            "surveillance",
            "citizen privacy",
        ),
        paper="Paper 2",
        source_pages=(33,),
        parent_concept_id="aqa_3_8_impacts",
    ),
    _concept(
        concept_id="aqa_3_8_contexts",
        official_reference="3.8",
        chapter_reference="3.8",
        chapter_title=AQA_CHAPTERS["3.8"],
        official_title=AQA_CHAPTERS["3.8"],
        label="Societal impact contexts",
        description=(
            "Apply ethical, legal, environmental and privacy principles to "
            "cyber security, mobile and wireless technologies, cloud "
            "storage, hacking, wearables, implants and autonomous vehicles."
        ),
        aliases=(
            "mobile technology impact",
            "wireless networking impact",
            "cloud storage impact",
            "hacking impact",
            "wearable technology",
            "computer implant",
            "autonomous vehicle",
            "self driving car",
        ),
        paper="Paper 2",
        source_pages=(33,),
        parent_concept_id="aqa_3_8_impacts",
    ),
)


# =============================================================================
# LOOKUP INDEXES
# =============================================================================

CONCEPT_BY_ID: dict[str, CSConcept] = {
    concept.concept_id: concept
    for concept in CS_CONCEPTS
}


CONCEPTS_BY_REFERENCE: dict[str, tuple[CSConcept, ...]] = {
    reference: tuple(concepts)
    for reference, concepts in (
        lambda grouped: grouped.items()
    )(
        (
            lambda grouped: [
                grouped[concept.official_reference].append(concept)
                for concept in CS_CONCEPTS
            ]
            and grouped
        )(
            defaultdict(list)
        )
    )
}


CONCEPTS_BY_CHAPTER: dict[str, tuple[CSConcept, ...]] = {
    chapter_reference: tuple(concepts)
    for chapter_reference, concepts in (
        lambda grouped: grouped.items()
    )(
        (
            lambda grouped: [
                grouped[concept.chapter_reference].append(concept)
                for concept in CS_CONCEPTS
            ]
            and grouped
        )(
            defaultdict(list)
        )
    )
}


def get_concept(concept_id: str) -> CSConcept:
    """
    Return one catalogue item by its stable internal concept ID.
    """

    try:
        return CONCEPT_BY_ID[concept_id]

    except KeyError as exc:
        raise KeyError(
            f"Unknown AQA concept_id: {concept_id}"
        ) from exc


def get_concepts_by_reference(
    official_reference: str,
) -> tuple[CSConcept, ...]:
    """
    Return all searchable concepts linked to one official AQA section.
    """

    return CONCEPTS_BY_REFERENCE.get(
        official_reference,
        (),
    )


def get_concepts_by_chapter(
    chapter_reference: str,
) -> tuple[CSConcept, ...]:
    """
    Return all searchable concepts inside an official AQA chapter.
    """

    return CONCEPTS_BY_CHAPTER.get(
        chapter_reference,
        (),
    )


def validate_catalog() -> None:
    """
    Fail early if the catalogue contains structural errors.
    """

    concept_ids: set[str] = set()

    for concept in CS_CONCEPTS:
        if concept.concept_id in concept_ids:
            raise ValueError(
                f"Duplicate concept_id: {concept.concept_id}"
            )

        concept_ids.add(
            concept.concept_id
        )

        if concept.chapter_reference not in AQA_CHAPTERS:
            raise ValueError(
                "Unknown chapter reference "
                f"{concept.chapter_reference} "
                f"for {concept.concept_id}"
            )

        if (
            concept.chapter_title
            != AQA_CHAPTERS[
                concept.chapter_reference
            ]
        ):
            raise ValueError(
                "Chapter title mismatch for "
                f"{concept.concept_id}"
            )

        if not concept.official_reference.startswith(
            concept.chapter_reference
        ):
            raise ValueError(
                "Official reference does not belong to chapter for "
                f"{concept.concept_id}"
            )

        if not concept.aliases:
            raise ValueError(
                f"No aliases defined for {concept.concept_id}"
            )

        for pattern in concept.match_patterns:
            if not pattern.label.strip():
                raise ValueError(
                    f"Empty pattern label for {concept.concept_id}"
                )

            if not 0.0 <= pattern.weight <= 1.0:
                raise ValueError(
                    f"Invalid pattern weight for {concept.concept_id}"
                )

            try:
                re.compile(pattern.regex, re.IGNORECASE)
            except re.error as exc:
                raise ValueError(
                    f"Invalid regex pattern for {concept.concept_id}: "
                    f"{pattern.regex}"
                ) from exc

        if concept.parent_concept_id is not None:
            if concept.parent_concept_id == concept.concept_id:
                raise ValueError(
                    f"Concept cannot parent itself: {concept.concept_id}"
                )

    missing_parents = {
        concept.parent_concept_id
        for concept in CS_CONCEPTS
        if (
            concept.parent_concept_id is not None
            and concept.parent_concept_id not in concept_ids
        )
    }

    if missing_parents:
        raise ValueError(
            "Missing parent concepts: "
            + ", ".join(
                sorted(missing_parents)
            )
        )

    missing_chapters = (
        set(AQA_CHAPTERS)
        - set(CONCEPTS_BY_CHAPTER)
    )

    if missing_chapters:
        raise ValueError(
            "No catalogue concepts for chapters: "
            + ", ".join(
                sorted(missing_chapters)
            )
        )


validate_catalog()

## Official-topic candidate extractor

**Exact source:** `app/services/topic_candidate_extractor.py`

The following cell contains the current source code without notebook-specific changes.

In [ ]:
from __future__ import annotations

import re
from collections.abc import Callable, Sequence
from dataclasses import dataclass

import numpy as np

from app.schemas.topic import RawTopicCandidate
from app.services.cs_concept_catalog import CS_CONCEPTS, CSConcept
from app.services.embedding_service import (
    CHUNKING_EMBEDDING_MODEL,
    embed_texts,
)


EmbeddingFunction = Callable[[Sequence[str], str, int], np.ndarray]


@dataclass(frozen=True)
class TopicExtractionConfig:
    """
    Configuration for official AQA topic candidate extraction.

    The salience rules are generic. They do not contain transcript-specific
    words. Their purpose is to stop an isolated ordinary word such as
    "integer", "function" or "bit" from becoming a lesson topic unless the
    surrounding semantic or repeated evidence supports it.
    """

    semantic_unit_words: int = 60

    raw_candidate_floor: float = 0.30
    semantic_only_threshold: float = 0.50

    # A topic supported only by one-word aliases requires stronger context.
    single_word_semantic_floor: float = 0.45
    single_word_min_evidence_sentences: int = 2
    single_word_min_distinct_aliases: int = 2

    max_raw_candidates: int = 12
    max_final_candidates: int = 6
    max_evidence_per_candidate: int = 3

    suppress_redundant_parents: bool = True
    parent_suppression_margin: float = 0.08

    # Suppress two labels that are driven by effectively the same sentence
    # and the same technical phrase.
    duplicate_evidence_overlap: float = 0.80
    duplicate_alias_token_overlap: float = 0.50

    embedding_model: str = CHUNKING_EMBEDDING_MODEL

    def __post_init__(self) -> None:
        if self.semantic_unit_words <= 0:
            raise ValueError("semantic_unit_words must be positive.")

        for value_name in (
            "raw_candidate_floor",
            "semantic_only_threshold",
            "single_word_semantic_floor",
            "parent_suppression_margin",
            "duplicate_evidence_overlap",
            "duplicate_alias_token_overlap",
        ):
            value = getattr(self, value_name)
            if not 0.0 <= value <= 1.0:
                raise ValueError(f"{value_name} must be between 0 and 1.")

        if self.max_raw_candidates < 1 or self.max_final_candidates < 1:
            raise ValueError("Candidate limits must be at least 1.")


@dataclass(frozen=True)
class KeywordEvidence:
    score: float
    matched_aliases: list[str]
    evidence_sentences: list[str]
    total_hits: int
    excluded_hits: int
    single_word_alias_only: bool


class TopicCandidateExtractor:
    """
    Extract official AQA topic candidates from one transcript chunk.

    Evidence sources:
    - exact transcript-friendly catalogue aliases
    - MiniLM semantic similarity
    - evidence repetition/diversity (topic salience)
    """

    def __init__(
        self,
        config: TopicExtractionConfig | None = None,
        embedding_function: EmbeddingFunction = embed_texts,
    ) -> None:
        self.config = config or TopicExtractionConfig()
        self._embedding_function = embedding_function
        self._concept_embeddings: np.ndarray | None = None

    def extract(
        self,
        chunk_id: int,
        text: str,
    ) -> list[RawTopicCandidate]:
        if chunk_id < 1:
            raise ValueError("chunk_id must be at least 1.")

        if not isinstance(text, str):
            raise TypeError("text must be a string.")

        text = text.strip()
        if not text:
            return []

        sentences = self._split_sentences(text)
        semantic_units = self._build_semantic_units(sentences) or [text]

        unit_embeddings = self._embedding_function(
            semantic_units,
            self.config.embedding_model,
            32,
        )
        concept_embeddings = self._get_concept_embeddings()

        if unit_embeddings.size == 0 or concept_embeddings.size == 0:
            return []

        similarity_matrix = unit_embeddings @ concept_embeddings.T
        raw_candidates: list[RawTopicCandidate] = []

        for concept_index, concept in enumerate(CS_CONCEPTS):
            similarities = similarity_matrix[:, concept_index]
            best_unit_index = int(np.argmax(similarities))
            semantic_score = float(similarities[best_unit_index])

            keyword = self._keyword_evidence(
                concept=concept,
                sentences=sentences,
                full_text=text,
            )

            has_keyword_support = keyword.score > 0.0
            # A known longer compound term can contain a shorter alias.
            # When every lexical hit was blocked by catalogue exclusions,
            # semantic similarity alone must not remap that compound back to
            # the shorter official concept.
            has_semantic_only_support = (
                semantic_score >= self.config.semantic_only_threshold
                and keyword.excluded_hits == 0
            )

            if not (has_keyword_support or has_semantic_only_support):
                continue

            if has_keyword_support and not self._passes_salience_gate(
                keyword=keyword,
                semantic_score=semantic_score,
            ):
                continue

            salience_score = self._calculate_salience_score(
                keyword=keyword,
                semantic_score=semantic_score,
            )

            confidence = self._calculate_confidence(
                keyword_score=keyword.score,
                semantic_score=semantic_score,
                single_word_alias_only=keyword.single_word_alias_only,
                salience_score=salience_score,
            )

            if confidence < self.config.raw_candidate_floor:
                continue

            semantic_evidence = semantic_units[best_unit_index].strip()
            evidence = (
                keyword.evidence_sentences
                if keyword.evidence_sentences
                else [semantic_evidence]
            )
            evidence = self._unique_strings(evidence)[
                : self.config.max_evidence_per_candidate
            ]

            if keyword.score > 0.0 and semantic_score >= 0.20:
                method = "keyword_embedding"
            elif keyword.score > 0.0:
                method = "keyword"
            else:
                method = "embedding"

            raw_candidates.append(
                RawTopicCandidate(
                    concept_id=concept.concept_id,
                    topic=concept.label,
                    domain=concept.domain,
                    official_reference=concept.official_reference,
                    chapter_reference=concept.chapter_reference,
                    official_title=concept.official_title,
                    paper=concept.paper,
                    source_pages=list(concept.source_pages),
                    confidence=round(confidence, 4),
                    keyword_score=round(keyword.score, 4),
                    semantic_score=round(semantic_score, 4),
                    salience_score=round(salience_score, 4),
                    extraction_method=method,
                    matched_aliases=keyword.matched_aliases,
                    total_alias_hits=keyword.total_hits,
                    evidence_sentence_count=len(keyword.evidence_sentences),
                    single_word_alias_only=keyword.single_word_alias_only,
                    evidence=evidence,
                    parent_concept_id=concept.parent_concept_id,
                )
            )

        raw_candidates.sort(
            key=lambda candidate: (
                candidate.confidence,
                candidate.salience_score,
                candidate.semantic_score,
                candidate.keyword_score,
            ),
            reverse=True,
        )

        raw_candidates = raw_candidates[: self.config.max_raw_candidates]

        if self.config.suppress_redundant_parents:
            raw_candidates = self._suppress_redundant_candidates(
                raw_candidates
            )

        return raw_candidates[: self.config.max_final_candidates]

    def _get_concept_embeddings(self) -> np.ndarray:
        if self._concept_embeddings is not None:
            return self._concept_embeddings

        concept_texts = [concept.embedding_text for concept in CS_CONCEPTS]
        self._concept_embeddings = self._embedding_function(
            concept_texts,
            self.config.embedding_model,
            32,
        )
        return self._concept_embeddings

    @staticmethod
    def _split_sentences(text: str) -> list[str]:
        text = re.sub(r"\s+", " ", text).strip()
        parts = re.split(r"(?<=[.!?])\s+", text)
        return [part.strip() for part in parts if part.strip()]

    def _build_semantic_units(self, sentences: list[str]) -> list[str]:
        units: list[str] = []
        buffer: list[str] = []
        buffer_words = 0

        for sentence in sentences:
            buffer.append(sentence)
            buffer_words += self._word_count(sentence)

            if buffer_words >= self.config.semantic_unit_words:
                units.append(" ".join(buffer))
                buffer = []
                buffer_words = 0

        if buffer:
            units.append(" ".join(buffer))

        return units

    def _keyword_evidence(
        self,
        concept: CSConcept,
        sentences: list[str],
        full_text: str,
    ) -> KeywordEvidence:
        matched_aliases: list[str] = []
        evidence: list[str] = []
        alias_weights: list[float] = []
        total_hits = 0
        excluded_hits = 0
        matched_word_counts: list[int] = []

        for alias in concept.aliases:
            normalized_alias = self._normalize_for_match(alias)
            if not normalized_alias:
                continue

            pattern = re.compile(
                r"(?<!\w)"
                + re.escape(normalized_alias).replace(r"\ ", r"\s+")
                + r"(?!\w)",
                re.IGNORECASE,
            )

            valid_alias_hits = 0

            for sentence in sentences:
                normalized_sentence = self._normalize_for_match(sentence)
                blocked_ranges = self._excluded_ranges(
                    text=normalized_sentence,
                    excluded_phrases=concept.excluded_phrases,
                )

                sentence_valid_hits = 0

                for match in pattern.finditer(normalized_sentence):
                    if self._overlaps_any(match.span(), blocked_ranges):
                        excluded_hits += 1
                        continue

                    sentence_valid_hits += 1

                if sentence_valid_hits:
                    valid_alias_hits += sentence_valid_hits
                    evidence.append(sentence.strip())

            if not valid_alias_hits:
                continue

            total_hits += valid_alias_hits
            matched_aliases.append(alias)

            alias_word_count = len(normalized_alias.split())
            matched_word_counts.append(alias_word_count)

            if alias_word_count == 1:
                weight = 0.50
            elif alias_word_count == 2:
                weight = 0.72
            else:
                weight = 0.82

            alias_weights.append(weight)

        # Flexible patterns support natural classroom phrasing such as
        # passive voice, inserted variable names and ASR variation. The
        # mechanism is catalogue-driven and can be reused by any concept.
        for flexible_pattern in concept.match_patterns:
            pattern = re.compile(
                flexible_pattern.regex,
                re.IGNORECASE,
            )

            pattern_hits = 0

            for sentence in sentences:
                normalized_sentence = self._normalize_for_match(sentence)
                blocked_ranges = self._excluded_ranges(
                    text=normalized_sentence,
                    excluded_phrases=concept.excluded_phrases,
                )

                sentence_hits = 0

                for match in pattern.finditer(normalized_sentence):
                    if self._overlaps_any(match.span(), blocked_ranges):
                        excluded_hits += 1
                        continue

                    sentence_hits += 1

                if sentence_hits:
                    pattern_hits += sentence_hits
                    evidence.append(sentence.strip())

            if not pattern_hits:
                continue

            total_hits += pattern_hits
            matched_aliases.append(flexible_pattern.label)
            matched_word_counts.append(
                max(2, len(flexible_pattern.label.split()))
            )
            alias_weights.append(flexible_pattern.weight)

        if not alias_weights:
            return KeywordEvidence(
                score=0.0,
                matched_aliases=[],
                evidence_sentences=[],
                total_hits=0,
                excluded_hits=excluded_hits,
                single_word_alias_only=False,
            )

        evidence = self._unique_strings(evidence)
        matched_aliases = self._unique_strings(matched_aliases)

        strongest_alias = max(alias_weights)
        distinct_bonus = 0.08 * max(0, len(matched_aliases) - 1)
        evidence_bonus = 0.05 * min(3, max(0, len(evidence) - 1))
        repetition_bonus = 0.02 * min(4, max(0, total_hits - 1))

        keyword_score = min(
            1.0,
            strongest_alias
            + distinct_bonus
            + evidence_bonus
            + repetition_bonus,
        )

        return KeywordEvidence(
            score=keyword_score,
            matched_aliases=matched_aliases,
            evidence_sentences=evidence,
            total_hits=total_hits,
            excluded_hits=excluded_hits,
            single_word_alias_only=all(
                word_count == 1 for word_count in matched_word_counts
            ),
        )

    @classmethod
    def _excluded_ranges(
        cls,
        text: str,
        excluded_phrases: tuple[str, ...],
    ) -> list[tuple[int, int]]:
        """
        Return character ranges occupied by longer confusable phrases.

        The rule is catalogue-driven rather than transcript-specific. Any
        concept can declare compound phrases which must not be consumed by a
        shorter alias.
        """

        ranges: list[tuple[int, int]] = []

        for phrase in excluded_phrases:
            normalized_phrase = cls._normalize_for_match(phrase)
            if not normalized_phrase:
                continue

            pattern = re.compile(
                r"(?<!\w)"
                + re.escape(normalized_phrase).replace(r"\ ", r"\s+")
                + r"(?!\w)",
                re.IGNORECASE,
            )

            ranges.extend(
                match.span()
                for match in pattern.finditer(text)
            )

        return ranges

    @staticmethod
    def _overlaps_any(
        span: tuple[int, int],
        blocked_ranges: list[tuple[int, int]],
    ) -> bool:
        start, end = span

        return any(
            start < blocked_end and end > blocked_start
            for blocked_start, blocked_end in blocked_ranges
        )

    def _passes_salience_gate(
        self,
        keyword: KeywordEvidence,
        semantic_score: float,
    ) -> bool:
        if not keyword.single_word_alias_only:
            return True

        return any(
            (
                semantic_score >= self.config.single_word_semantic_floor,
                len(keyword.evidence_sentences)
                >= self.config.single_word_min_evidence_sentences,
                len(keyword.matched_aliases)
                >= self.config.single_word_min_distinct_aliases,
            )
        )

    @staticmethod
    def _calculate_salience_score(
        keyword: KeywordEvidence,
        semantic_score: float,
    ) -> float:
        semantic_component = max(0.0, min(1.0, semantic_score))
        evidence_component = min(1.0, len(keyword.evidence_sentences) / 3.0)
        alias_component = min(1.0, len(keyword.matched_aliases) / 2.0)

        return min(
            1.0,
            0.55 * semantic_component
            + 0.25 * evidence_component
            + 0.20 * alias_component,
        )

    @staticmethod
    def _calculate_confidence(
        keyword_score: float,
        semantic_score: float,
        single_word_alias_only: bool,
        salience_score: float,
    ) -> float:
        semantic_component = max(0.0, min(1.0, semantic_score))

        combined = 0.58 * semantic_component + 0.42 * keyword_score

        # Multiword technical phrases are stronger direct evidence than an
        # isolated one-word match. Single-word matches therefore cannot win
        # through the keyword path alone.
        keyword_multiplier = 0.72 if single_word_alias_only else 0.88
        keyword_supported = keyword_multiplier * keyword_score
        semantic_only = 0.82 * semantic_component

        base_confidence = max(combined, keyword_supported, semantic_only)

        # Salience has a bounded effect: it can reduce an incidental mention,
        # but it cannot manufacture confidence without real evidence.
        adjusted = base_confidence * (0.85 + 0.15 * salience_score)

        return max(0.0, min(1.0, adjusted))

    def _suppress_redundant_candidates(
        self,
        candidates: list[RawTopicCandidate],
    ) -> list[RawTopicCandidate]:
        kept: list[RawTopicCandidate] = []

        for candidate in candidates:
            should_skip = False

            for existing in kept:
                if self._is_redundant_pair(candidate, existing):
                    should_skip = True
                    break

            if not should_skip:
                kept.append(candidate)

        return kept

    def _is_redundant_pair(
        self,
        candidate: RawTopicCandidate,
        existing: RawTopicCandidate,
    ) -> bool:
        # Parent/child suppression.
        if candidate.parent_concept_id == existing.concept_id:
            return candidate.confidence <= (
                existing.confidence + self.config.parent_suppression_margin
            )

        if existing.parent_concept_id == candidate.concept_id:
            return candidate.confidence <= existing.confidence

        evidence_overlap = self._jaccard(
            self._normalized_evidence(candidate.evidence),
            self._normalized_evidence(existing.evidence),
        )

        alias_overlap = self._jaccard(
            self._alias_tokens(candidate.matched_aliases),
            self._alias_tokens(existing.matched_aliases),
        )

        same_reference = (
            candidate.official_reference == existing.official_reference
        )

        return (
            evidence_overlap >= self.config.duplicate_evidence_overlap
            and alias_overlap >= self.config.duplicate_alias_token_overlap
            and (same_reference or self._topic_token_overlap(candidate, existing) >= 0.50)
        )

    def _topic_token_overlap(
        self,
        first: RawTopicCandidate,
        second: RawTopicCandidate,
    ) -> float:
        return self._jaccard(
            self._stemmed_tokens(first.topic),
            self._stemmed_tokens(second.topic),
        )

    @classmethod
    def _alias_tokens(cls, aliases: list[str]) -> set[str]:
        tokens: set[str] = set()
        for alias in aliases:
            tokens.update(cls._stemmed_tokens(alias))
        return tokens

    @classmethod
    def _normalized_evidence(cls, evidence: list[str]) -> set[str]:
        return {
            cls._normalize_for_match(value)
            for value in evidence
            if cls._normalize_for_match(value)
        }

    @classmethod
    def _stemmed_tokens(cls, text: str) -> set[str]:
        tokens = cls._normalize_for_match(text).split()
        return {cls._light_stem(token) for token in tokens if len(token) > 2}

    @staticmethod
    def _light_stem(token: str) -> str:
        for suffix in ("ing", "ed", "es", "s"):
            if token.endswith(suffix) and len(token) > len(suffix) + 3:
                return token[: -len(suffix)]
        return token

    @staticmethod
    def _jaccard(first: set[str], second: set[str]) -> float:
        if not first or not second:
            return 0.0
        return len(first & second) / len(first | second)

    @staticmethod
    def _normalize_for_match(text: str) -> str:
        text = text.lower()
        text = re.sub(r"[^a-z0-9]+", " ", text)
        return re.sub(r"\s+", " ", text).strip()

    @staticmethod
    def _word_count(text: str) -> int:
        return len(re.findall(r"\S+", text))

    @staticmethod
    def _unique_strings(values: list[str]) -> list[str]:
        unique: list[str] = []
        seen: set[str] = set()

        for value in values:
            normalized = re.sub(r"\s+", " ", value).strip().lower()
            if not normalized or normalized in seen:
                continue
            seen.add(normalized)
            unique.append(value.strip())

        return unique

## CS relevance filter

**Exact source:** `app/services/cs_relevance_filter.py`

The following cell contains the current source code without notebook-specific changes.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass

from app.schemas.topic import (
    ChunkTopicResult,
    RawTopicCandidate,
    TopicCandidate,
)


@dataclass(frozen=True)
class CSRelevanceConfig:
    """
    Thresholds for retaining official AQA topic candidates.
    """

    candidate_keep_threshold: float = 0.46
    uncertain_candidate_floor: float = 0.34
    llm_fallback_min_words: int = 80
    max_rejected_candidates: int = 3

    def __post_init__(self) -> None:
        if not 0.0 <= self.candidate_keep_threshold <= 1.0:
            raise ValueError(
                "candidate_keep_threshold must be between 0 and 1."
            )

        if not 0.0 <= self.uncertain_candidate_floor <= 1.0:
            raise ValueError(
                "uncertain_candidate_floor must be between 0 and 1."
            )

        if self.uncertain_candidate_floor > self.candidate_keep_threshold:
            raise ValueError(
                "uncertain_candidate_floor cannot exceed "
                "candidate_keep_threshold."
            )


class CSRelevanceFilter:
    """
    Keep strong official AQA candidates and retain low-confidence candidates
    separately for inspection.
    """

    def __init__(self, config: CSRelevanceConfig | None = None) -> None:
        self.config = config or CSRelevanceConfig()

    def filter(
        self,
        chunk_id: int,
        source_word_count: int,
        candidates: list[RawTopicCandidate],
    ) -> ChunkTopicResult:
        relevant: list[TopicCandidate] = []
        rejected: list[TopicCandidate] = []

        for candidate in candidates:
            relevance_score = candidate.confidence
            is_relevant = (
                relevance_score >= self.config.candidate_keep_threshold
            )

            filtered_candidate = TopicCandidate(
                **candidate.model_dump(),
                cs_relevance_score=round(relevance_score, 4),
                cs_relevant=is_relevant,
            )

            if is_relevant:
                relevant.append(filtered_candidate)
            else:
                rejected.append(filtered_candidate)

        relevant.sort(
            key=lambda candidate: (
                candidate.cs_relevance_score,
                candidate.salience_score,
            ),
            reverse=True,
        )
        rejected.sort(
            key=lambda candidate: candidate.cs_relevance_score,
            reverse=True,
        )

        best_score = 0.0

        if relevant:
            best_score = max(
                candidate.cs_relevance_score for candidate in relevant
            )
            # Multiple topics increase chunk-level confidence only slightly.
            support_bonus = min(0.06, 0.02 * max(0, len(relevant) - 1))
            chunk_relevance_score = min(0.95, best_score + support_bonus)
        elif rejected:
            best_score = max(
                candidate.cs_relevance_score for candidate in rejected
            )
            chunk_relevance_score = best_score
        else:
            chunk_relevance_score = 0.0

        requires_llm_fallback = (
            not relevant
            and source_word_count >= self.config.llm_fallback_min_words
            and best_score >= self.config.uncertain_candidate_floor
        )

        notes: list[str] = []
        if not candidates:
            notes.append("No official AQA topic candidate was detected.")
        elif not relevant:
            notes.append(
                "Only low-confidence official AQA candidates were detected."
            )

        if requires_llm_fallback:
            notes.append(
                "Chunk is borderline and may require GPT-OSS fallback."
            )

        return ChunkTopicResult(
            chunk_id=chunk_id,
            source_word_count=source_word_count,
            classification=(
                "official_aqa_topic" if relevant else "no_topic"
            ),
            is_cs_relevant=bool(relevant),
            creates_new_topic=bool(relevant),
            cs_relevance_score=round(chunk_relevance_score, 4),
            topic_candidates=relevant,
            rejected_candidates=rejected[
                : self.config.max_rejected_candidates
            ],
            requires_llm_fallback=requires_llm_fallback,
            notes=notes,
        )

## Unmapped CS detector

**Exact source:** `app/services/cs_unmapped_detector.py`

The following cell contains the current source code without notebook-specific changes.

In [ ]:
from __future__ import annotations

import re
from collections.abc import Callable, Sequence
from dataclasses import dataclass

import numpy as np

from app.schemas.topic import TopicCandidate, UnmappedCSSignal
from app.services.embedding_service import (
    CHUNKING_EMBEDDING_MODEL,
    embed_texts,
)


EmbeddingFunction = Callable[[Sequence[str], str, int], np.ndarray]


@dataclass(frozen=True)
class CSDomain:
    name: str
    description: str


@dataclass(frozen=True)
class UnmappedConceptFamily:
    """
    A generic CS concept family which is useful for detecting syllabus gaps.

    These entries are not official AQA mappings. They only provide a rough
    label when technical content remains outside the official catalogue.
    """

    rough_topic: str
    domain: str
    description: str
    aliases: tuple[str, ...]


# Broad descriptions used only for residual semantic detection.
CS_DOMAINS: tuple[CSDomain, ...] = (
    CSDomain(
        name="Programming and software development",
        description=(
            "Programming source code, variables, control flow, methods, "
            "classes, objects, constructors, attributes, access control, "
            "data structures and software behaviour."
        ),
    ),
    CSDomain(
        name="Algorithms and computational thinking",
        description=(
            "Algorithms, problem solving, tracing execution, searching, "
            "sorting, decomposition, abstraction and efficiency."
        ),
    ),
    CSDomain(
        name="Data representation",
        description=(
            "Binary, hexadecimal, bits, bytes, text encoding, images, "
            "sound, file size and compression."
        ),
    ),
    CSDomain(
        name="Computer systems",
        description=(
            "Computer hardware, software, CPU architecture, memory, "
            "storage, operating systems and translators."
        ),
    ),
    CSDomain(
        name="Networks and cyber security",
        description=(
            "Computer networks, protocols, network devices, security, "
            "attacks, authentication, encryption and protective controls."
        ),
    ),
    CSDomain(
        name="Databases and data management",
        description=(
            "Relational databases, tables, records, fields, keys, SQL and "
            "database queries."
        ),
    ),
)


# Generic, transcript-independent families commonly encountered in lessons
# but not represented as explicit AQA 8525 catalogue topics.
UNMAPPED_CONCEPT_FAMILIES: tuple[UnmappedConceptFamily, ...] = (
    UnmappedConceptFamily(
        rough_topic="Dynamic arrays and list collections",
        domain="Programming and software development",
        description=(
            "Resizable sequence data structures such as ArrayList or a "
            "dynamic array whose size can change while a program runs."
        ),
        aliases=(
            "array list",
            "array lists",
            "arraylist",
            "arraylists",
            "dynamic array",
            "resizable array",
            "resizable list",
        ),
    ),
    UnmappedConceptFamily(
        rough_topic="Object construction and initialisation",
        domain="Programming and software development",
        description=(
            "Constructors create or initialise objects and set their "
            "initial attributes or state."
        ),
        aliases=(
            "constructor",
            "constructors",
            "object constructor",
            "create an object",
            "initialise an object",
            "initialize an object",
            "object initialisation",
            "object initialization",
            "class instance",
        ),
    ),
    UnmappedConceptFamily(
        rough_topic="Encapsulation and access modifiers",
        domain="Programming and software development",
        description=(
            "Private and public members, class attributes and controlled "
            "access to an object's internal state."
        ),
        aliases=(
            "private attribute",
            "private attributes",
            "public attribute",
            "public attributes",
            "private and public",
            "access modifier",
            "access modifiers",
            "outside the class",
            "encapsulation",
            "controlled access",
        ),
    ),
    UnmappedConceptFamily(
        rough_topic="Inheritance and polymorphism",
        domain="Programming and software development",
        description=(
            "Object-oriented inheritance, subclasses, method overriding "
            "and polymorphic behaviour."
        ),
        aliases=(
            "inheritance",
            "subclass",
            "superclass",
            "base class",
            "derived class",
            "method overriding",
            "polymorphism",
        ),
    ),
)


@dataclass(frozen=True)
class UnmappedDetectionConfig:
    sentence_similarity_threshold: float = 0.48
    strong_sentence_threshold: float = 0.60
    minimum_evidence_sentences: int = 2
    max_signals: int = 4
    embedding_model: str = CHUNKING_EMBEDDING_MODEL


class CSUnmappedDetector:
    """
    Detect CS content which is not explained by retained official topics.

    Detection has two generic routes:
    1. lexical families for recognisable off-syllabus concepts;
    2. broad semantic residual detection for unknown CS material.

    The detector never fabricates an official AQA reference.
    """

    def __init__(
        self,
        config: UnmappedDetectionConfig | None = None,
        embedding_function: EmbeddingFunction = embed_texts,
    ) -> None:
        self.config = config or UnmappedDetectionConfig()
        self._embedding_function = embedding_function
        self._domain_embeddings: np.ndarray | None = None
        self._family_embeddings: np.ndarray | None = None

    def detect(
        self,
        text: str,
        official_candidates: list[TopicCandidate],
    ) -> list[UnmappedCSSignal]:
        sentences = self._split_sentences(text)
        if not sentences:
            return []

        lexical_signals = self._detect_known_families(
            sentences=sentences,
            official_candidates=official_candidates,
        )

        semantic_signals = self._detect_semantic_residual(
            sentences=sentences,
            official_candidates=official_candidates,
        )

        # A specific lexical family already gives a clearer rough topic than
        # the generic semantic residual for the same sentence. Keeping both
        # would create duplicate evidence and could trigger an unnecessary
        # LLM fallback.
        specific_evidence = {
            self._normalize(signal.evidence)
            for signal in lexical_signals
        }
        semantic_signals = [
            signal
            for signal in semantic_signals
            if self._normalize(signal.evidence) not in specific_evidence
        ]

        combined = lexical_signals + semantic_signals
        combined.sort(key=lambda signal: signal.score, reverse=True)

        return self._deduplicate(combined)[: self.config.max_signals]

    def _detect_known_families(
        self,
        sentences: list[str],
        official_candidates: list[TopicCandidate],
    ) -> list[UnmappedCSSignal]:
        official_terms = self._official_terms(official_candidates)
        family_embeddings = self._get_family_embeddings()
        signals: list[UnmappedCSSignal] = []

        for family_index, family in enumerate(UNMAPPED_CONCEPT_FAMILIES):
            matched_aliases: list[str] = []
            evidence_sentences: list[str] = []

            for sentence in sentences:
                normalized_sentence = self._normalize(sentence)
                sentence_aliases = [
                    alias
                    for alias in family.aliases
                    if self._contains_phrase(
                        normalized_sentence,
                        self._normalize(alias),
                    )
                ]

                if not sentence_aliases:
                    continue

                matched_aliases.extend(sentence_aliases)
                evidence_sentences.append(sentence)

            matched_aliases = self._unique_strings(matched_aliases)
            evidence_sentences = self._unique_strings(evidence_sentences)

            if not matched_aliases:
                continue

            # Do not duplicate an official candidate that already uses the
            # same specific terminology. Partial overlap such as array versus
            # ArrayList is deliberately not treated as coverage.
            if self._family_is_officially_covered(
                family=family,
                matched_aliases=matched_aliases,
                official_terms=official_terms,
            ):
                continue

            evidence = evidence_sentences[0]
            evidence_embedding = self._embedding_function(
                [evidence],
                self.config.embedding_model,
                32,
            )
            semantic_score = float(
                evidence_embedding[0] @ family_embeddings[family_index]
            )

            longest_alias_words = max(
                len(self._normalize(alias).split())
                for alias in matched_aliases
            )

            lexical_base = 0.62 if longest_alias_words >= 2 else 0.56
            diversity_bonus = 0.05 * min(3, len(matched_aliases) - 1)
            evidence_bonus = 0.04 * min(2, len(evidence_sentences) - 1)

            score = min(
                0.95,
                lexical_base
                + diversity_bonus
                + evidence_bonus
                + 0.18 * max(0.0, semantic_score),
            )

            method = (
                "lexical_semantic"
                if semantic_score >= self.config.sentence_similarity_threshold
                else "lexical"
            )

            signals.append(
                UnmappedCSSignal(
                    rough_topic=family.rough_topic,
                    domain=family.domain,
                    score=round(score, 4),
                    evidence=evidence.strip(),
                    matched_aliases=matched_aliases,
                    detection_method=method,
                )
            )

        return signals

    def _detect_semantic_residual(
        self,
        sentences: list[str],
        official_candidates: list[TopicCandidate],
    ) -> list[UnmappedCSSignal]:
        covered = {
            self._normalize(sentence)
            for candidate in official_candidates
            for sentence in candidate.evidence
            if self._normalize(sentence)
        }

        uncovered_sentences = [
            sentence
            for sentence in sentences
            if self._normalize(sentence) not in covered
        ]

        if not uncovered_sentences:
            return []

        sentence_embeddings = self._embedding_function(
            uncovered_sentences,
            self.config.embedding_model,
            32,
        )
        domain_embeddings = self._get_domain_embeddings()

        similarities = sentence_embeddings @ domain_embeddings.T
        candidates: list[UnmappedCSSignal] = []

        for sentence_index, sentence in enumerate(uncovered_sentences):
            row = similarities[sentence_index]
            domain_index = int(np.argmax(row))
            score = float(row[domain_index])

            if score < self.config.sentence_similarity_threshold:
                continue

            candidates.append(
                UnmappedCSSignal(
                    rough_topic="Unmapped Computer Science content",
                    domain=CS_DOMAINS[domain_index].name,
                    score=round(score, 4),
                    evidence=sentence.strip(),
                    matched_aliases=[],
                    detection_method="semantic",
                )
            )

        candidates.sort(key=lambda signal: signal.score, reverse=True)
        candidates = self._deduplicate(candidates)

        strong_exists = any(
            signal.score >= self.config.strong_sentence_threshold
            for signal in candidates
        )

        if (
            len(candidates) < self.config.minimum_evidence_sentences
            and not strong_exists
        ):
            return []

        return candidates

    def _get_domain_embeddings(self) -> np.ndarray:
        if self._domain_embeddings is None:
            self._domain_embeddings = self._embedding_function(
                [domain.description for domain in CS_DOMAINS],
                self.config.embedding_model,
                32,
            )
        return self._domain_embeddings

    def _get_family_embeddings(self) -> np.ndarray:
        if self._family_embeddings is None:
            self._family_embeddings = self._embedding_function(
                [family.description for family in UNMAPPED_CONCEPT_FAMILIES],
                self.config.embedding_model,
                32,
            )
        return self._family_embeddings

    @classmethod
    def _official_terms(
        cls,
        official_candidates: list[TopicCandidate],
    ) -> set[str]:
        terms: set[str] = set()

        for candidate in official_candidates:
            terms.add(cls._normalize(candidate.topic))
            terms.add(cls._normalize(candidate.official_title))
            terms.update(
                cls._normalize(alias)
                for alias in candidate.matched_aliases
            )

        return {term for term in terms if term}

    @classmethod
    def _family_is_officially_covered(
        cls,
        family: UnmappedConceptFamily,
        matched_aliases: list[str],
        official_terms: set[str],
    ) -> bool:
        family_terms = {
            cls._normalize(family.rough_topic),
            *(
                cls._normalize(alias)
                for alias in matched_aliases
            ),
        }

        return any(
            family_term == official_term
            for family_term in family_terms
            for official_term in official_terms
            if family_term and official_term
        )

    @staticmethod
    def _split_sentences(text: str) -> list[str]:
        text = re.sub(r"\s+", " ", text).strip()
        parts = re.split(r"(?<=[.!?])\s+", text)
        return [part.strip() for part in parts if part.strip()]

    @staticmethod
    def _normalize(text: str) -> str:
        text = re.sub(r"[^a-z0-9]+", " ", text.lower())
        return re.sub(r"\s+", " ", text).strip()

    @staticmethod
    def _contains_phrase(text: str, phrase: str) -> bool:
        if not phrase:
            return False

        pattern = re.compile(
            r"(?<!\w)"
            + re.escape(phrase).replace(r"\ ", r"\s+")
            + r"(?!\w)",
            re.IGNORECASE,
        )
        return bool(pattern.search(text))

    @classmethod
    def _deduplicate(
        cls,
        signals: list[UnmappedCSSignal],
    ) -> list[UnmappedCSSignal]:
        output: list[UnmappedCSSignal] = []
        seen: set[tuple[str, str]] = set()

        for signal in signals:
            key = (
                cls._normalize(signal.rough_topic),
                cls._normalize(signal.evidence),
            )
            if not key[0] or not key[1] or key in seen:
                continue
            seen.add(key)
            output.append(signal)

        return output

    @staticmethod
    def _unique_strings(values: list[str]) -> list[str]:
        unique: list[str] = []
        seen: set[str] = set()

        for value in values:
            normalized = " ".join(value.lower().split())
            if not normalized or normalized in seen:
                continue
            seen.add(normalized)
            unique.append(value.strip())

        return unique

## Topic merger

**Exact source:** `app/services/topic_merger.py`

The following cell contains the current source code without notebook-specific changes.

In [ ]:
from __future__ import annotations

from collections import defaultdict
from dataclasses import dataclass
from statistics import fmean

from app.schemas.topic import ChunkTopicResult, MergedTopic, TopicCandidate


@dataclass(frozen=True)
class TopicMergeConfig:
    """
    Merge repeated official topics and rank lesson topics without allowing
    repeated low-semantic keyword matches to dominate the final order.
    """

    max_evidence_per_topic: int = 5

    # Only a new non-adjacent support span adds confidence.
    non_adjacent_span_bonus: float = 0.02
    maximum_support_bonus: float = 0.06
    maximum_merged_confidence: float = 0.95

    # Ranking combines independent signals. These weights do not change the
    # candidate acceptance threshold; they only order already-retained topics.
    ranking_confidence_weight: float = 0.30
    ranking_semantic_weight: float = 0.30
    ranking_salience_weight: float = 0.25
    ranking_coverage_weight: float = 0.15

    primary_min_semantic_score: float = 0.32
    primary_min_salience_score: float = 0.45
    primary_min_ranking_score: float = 0.48
    primary_min_coverage_score: float = 0.25


class TopicMerger:
    """
    Merge the same official concept across chunks.

    Consecutive chunk IDs are one support span because they may come from one
    long discussion split by Module 2's size guardrail.
    """

    def __init__(self, config: TopicMergeConfig | None = None) -> None:
        self.config = config or TopicMergeConfig()

    def merge(
        self,
        chunk_results: list[ChunkTopicResult],
    ) -> list[MergedTopic]:
        grouped: dict[
            str,
            list[tuple[int, TopicCandidate]],
        ] = defaultdict(list)

        topic_bearing_chunk_count = max(
            1,
            sum(
                1
                for chunk_result in chunk_results
                if chunk_result.topic_candidates
            ),
        )

        for chunk_result in chunk_results:
            for candidate in chunk_result.topic_candidates:
                grouped[candidate.concept_id].append(
                    (chunk_result.chunk_id, candidate)
                )

        merged_topics: list[MergedTopic] = []

        for concept_id, occurrences in grouped.items():
            first_candidate = occurrences[0][1]
            source_chunk_ids = sorted(
                {chunk_id for chunk_id, _ in occurrences}
            )

            support_span_count = self._count_contiguous_spans(
                source_chunk_ids
            )

            base_confidence = max(
                candidate.cs_relevance_score
                for _, candidate in occurrences
            )

            support_bonus = min(
                self.config.maximum_support_bonus,
                self.config.non_adjacent_span_bonus
                * max(0, support_span_count - 1),
            )

            confidence = min(
                self.config.maximum_merged_confidence,
                base_confidence + support_bonus,
            )

            mean_semantic_score = fmean(
                candidate.semantic_score
                for _, candidate in occurrences
            )
            mean_keyword_score = fmean(
                candidate.keyword_score
                for _, candidate in occurrences
            )
            mean_salience_score = fmean(
                candidate.salience_score
                for _, candidate in occurrences
            )

            coverage_score = min(
                1.0,
                len(source_chunk_ids) / topic_bearing_chunk_count,
            )

            ranking_score = self._ranking_score(
                confidence=confidence,
                mean_semantic_score=mean_semantic_score,
                mean_salience_score=mean_salience_score,
                coverage_score=coverage_score,
            )

            topic_role = self._topic_role(
                ranking_score=ranking_score,
                mean_semantic_score=mean_semantic_score,
                mean_salience_score=mean_salience_score,
                coverage_score=coverage_score,
                support_span_count=support_span_count,
                supporting_candidate_count=len(occurrences),
            )

            evidence: list[str] = []
            for _, candidate in occurrences:
                evidence.extend(candidate.evidence)

            merged_topics.append(
                MergedTopic(
                    concept_id=concept_id,
                    topic=first_candidate.topic,
                    domain=first_candidate.domain,
                    official_reference=first_candidate.official_reference,
                    chapter_reference=first_candidate.chapter_reference,
                    official_title=first_candidate.official_title,
                    paper=first_candidate.paper,
                    source_pages=first_candidate.source_pages,
                    confidence=round(confidence, 4),
                    ranking_score=round(ranking_score, 4),
                    topic_role=topic_role,
                    source_chunk_ids=source_chunk_ids,
                    support_span_count=support_span_count,
                    mean_semantic_score=round(mean_semantic_score, 4),
                    mean_keyword_score=round(mean_keyword_score, 4),
                    mean_salience_score=round(mean_salience_score, 4),
                    coverage_score=round(coverage_score, 4),
                    evidence=self._unique_strings(evidence)[
                        : self.config.max_evidence_per_topic
                    ],
                    supporting_candidate_count=len(occurrences),
                )
            )

        role_priority = {
            "primary": 1,
            "supporting": 0,
        }

        merged_topics.sort(
            key=lambda topic: (
                role_priority[topic.topic_role],
                topic.ranking_score,
                topic.mean_semantic_score,
                topic.confidence,
            ),
            reverse=True,
        )

        return merged_topics

    def _ranking_score(
        self,
        confidence: float,
        mean_semantic_score: float,
        mean_salience_score: float,
        coverage_score: float,
    ) -> float:
        semantic_component = max(
            0.0,
            min(1.0, mean_semantic_score),
        )

        score = (
            self.config.ranking_confidence_weight * confidence
            + self.config.ranking_semantic_weight * semantic_component
            + self.config.ranking_salience_weight * mean_salience_score
            + self.config.ranking_coverage_weight * coverage_score
        )

        return max(0.0, min(1.0, score))

    def _topic_role(
        self,
        ranking_score: float,
        mean_semantic_score: float,
        mean_salience_score: float,
        coverage_score: float,
        support_span_count: int,
        supporting_candidate_count: int,
    ) -> str:
        has_broad_lesson_support = any(
            (
                coverage_score >= self.config.primary_min_coverage_score,
                support_span_count >= 2,
                supporting_candidate_count >= 3,
            )
        )

        if (
            ranking_score >= self.config.primary_min_ranking_score
            and mean_semantic_score >= self.config.primary_min_semantic_score
            and mean_salience_score >= self.config.primary_min_salience_score
            and has_broad_lesson_support
        ):
            return "primary"

        return "supporting"

    @staticmethod
    def _count_contiguous_spans(chunk_ids: list[int]) -> int:
        if not chunk_ids:
            return 0

        spans = 1
        for previous, current in zip(chunk_ids, chunk_ids[1:]):
            if current > previous + 1:
                spans += 1
        return spans

    @staticmethod
    def _unique_strings(values: list[str]) -> list[str]:
        unique: list[str] = []
        seen: set[str] = set()

        for value in values:
            normalized = " ".join(value.lower().split())
            if not normalized or normalized in seen:
                continue
            seen.add(normalized)
            unique.append(value.strip())

        return unique

## Module 3 topic pipeline

**Exact source:** `app/services/topic_pipeline.py`

The following cell contains the current source code without notebook-specific changes.

In [ ]:
from __future__ import annotations

import re
from collections.abc import Sequence
from dataclasses import dataclass
from typing import Any

from app.schemas.topic import (
    ChunkTopicResult,
    Module3Result,
    UnmappedCSSignal,
)
from app.services.cs_relevance_filter import CSRelevanceFilter
from app.services.cs_unmapped_detector import CSUnmappedDetector
from app.services.topic_candidate_extractor import TopicCandidateExtractor
from app.services.topic_merger import TopicMerger


@dataclass(frozen=True)
class Module3PipelineConfig:
    """
    Decision rules for LLM fallback and continuation handling.

    Strong, specific lexical evidence can be retained directly. Semantic-only
    or ambiguous evidence is escalated because it requires interpretation.
    """

    strong_unmapped_lexical_score: float = 0.70
    strong_unmapped_semantic_score: float = 0.82
    ambiguous_signal_margin: float = 0.04

    def __post_init__(self) -> None:
        for field_name in (
            "strong_unmapped_lexical_score",
            "strong_unmapped_semantic_score",
            "ambiguous_signal_margin",
        ):
            value = getattr(self, field_name)
            if not 0.0 <= value <= 1.0:
                raise ValueError(
                    f"{field_name} must be between 0 and 1."
                )


class Module3TopicPipeline:
    """
    Complete Module 3 pipeline:

    Module 2 chunks
        → official AQA candidate extraction
        → salience-aware filtering
        → continuation/no-new-topic handling
        → unmapped CS detection
        → official topic merging
    """

    def __init__(
        self,
        extractor: TopicCandidateExtractor | None = None,
        relevance_filter: CSRelevanceFilter | None = None,
        unmapped_detector: CSUnmappedDetector | None = None,
        merger: TopicMerger | None = None,
        config: Module3PipelineConfig | None = None,
    ) -> None:
        self.extractor = extractor or TopicCandidateExtractor()
        self.relevance_filter = relevance_filter or CSRelevanceFilter()
        self.unmapped_detector = unmapped_detector or CSUnmappedDetector()
        self.merger = merger or TopicMerger()
        self.config = config or Module3PipelineConfig()

    def process_chunks(self, chunks: Sequence[Any]) -> Module3Result:
        chunk_results: list[ChunkTopicResult] = []

        for raw_chunk in chunks:
            chunk_id = int(self._get_value(raw_chunk, "chunk_id"))
            text = str(self._get_value(raw_chunk, "text")).strip()

            word_count_value = self._get_optional_value(
                raw_chunk,
                "word_count",
            )
            word_count = (
                int(word_count_value)
                if word_count_value is not None
                else len(re.findall(r"\S+", text))
            )

            overlap_word_count = int(
                self._get_optional_value(
                    raw_chunk,
                    "overlap_word_count",
                )
                or 0
            )

            raw_candidates = self.extractor.extract(
                chunk_id=chunk_id,
                text=text,
            )

            base_result = self.relevance_filter.filter(
                chunk_id=chunk_id,
                source_word_count=word_count,
                candidates=raw_candidates,
            )

            previous_result = chunk_results[-1] if chunk_results else None

            # A chunk created with overlap immediately after a retained CS
            # chunk may simply conclude the same question. It should not be
            # forced into a new topic or sent to the unmapped detector.
            current_topic_ids = {
                candidate.concept_id
                for candidate in base_result.topic_candidates
            }
            previous_topic_ids = self._effective_previous_topic_ids(
                previous_result=previous_result,
                completed_results=chunk_results,
            )

            continuation_only = (
                overlap_word_count > 0
                and previous_result is not None
                and bool(previous_topic_ids)
                and (
                    not current_topic_ids
                    or current_topic_ids.issubset(previous_topic_ids)
                )
            )

            if continuation_only:
                final_result = base_result.model_copy(
                    update={
                        "classification": "continuation_no_new_topic",
                        "is_cs_relevant": False,
                        "creates_new_topic": False,
                        "cs_relevance_score": 0.0,
                        "topic_candidates": [],
                        "has_unmapped_cs_content": False,
                        "unmapped_cs_signals": [],
                        "continuation_of_chunk_id": (
                            previous_result.continuation_of_chunk_id
                            or previous_result.chunk_id
                        ),
                        "requires_llm_fallback": False,
                        "notes": [
                            "No new standalone topic was detected; the chunk "
                            "continues the previous overlapped discussion."
                        ],
                    }
                )
                chunk_results.append(final_result)
                continue

            unmapped_signals = self.unmapped_detector.detect(
                text=text,
                official_candidates=base_result.topic_candidates,
            )
            has_unmapped = bool(unmapped_signals)
            has_official = bool(base_result.topic_candidates)

            if has_official and has_unmapped:
                classification = "mixed_official_and_unmapped"
            elif has_official:
                classification = "official_aqa_topic"
            elif has_unmapped:
                classification = "cs_related_unmapped"
            else:
                classification = "no_topic"

            unmapped_requires_llm = (
                self._unmapped_requires_llm_fallback(unmapped_signals)
            )

            requires_llm_fallback = (
                unmapped_requires_llm
                or (
                    base_result.requires_llm_fallback
                    and not has_unmapped
                )
            )

            notes = list(base_result.notes)
            if has_unmapped and unmapped_requires_llm:
                notes.append(
                    "Unmapped CS evidence is semantic-only, borderline or "
                    "ambiguous, so later fallback should refine the rough "
                    "topic without inventing an AQA label."
                )
            elif has_unmapped:
                notes.append(
                    "Strong specific unmapped-CS evidence was retained "
                    "directly; no LLM fallback is required."
                )

            final_score = base_result.cs_relevance_score
            if has_unmapped and not has_official:
                final_score = max(
                    signal.score for signal in unmapped_signals
                )

            final_result = base_result.model_copy(
                update={
                    "classification": classification,
                    "is_cs_relevant": has_official or has_unmapped,
                    "creates_new_topic": has_official or has_unmapped,
                    "cs_relevance_score": round(final_score, 4),
                    "has_unmapped_cs_content": has_unmapped,
                    "unmapped_cs_signals": unmapped_signals,
                    "requires_llm_fallback": requires_llm_fallback,
                    "notes": notes,
                }
            )

            chunk_results.append(final_result)

        merged_topics = self.merger.merge(chunk_results)

        classification_counts = {
            "official_aqa_topic": 0,
            "mixed_official_and_unmapped": 0,
            "cs_related_unmapped": 0,
            "continuation_no_new_topic": 0,
            "no_topic": 0,
        }

        for result in chunk_results:
            classification_counts[result.classification] += 1

        cs_relevant_chunks = sum(
            1 for result in chunk_results if result.is_cs_relevant
        )

        llm_fallback_ids = [
            result.chunk_id
            for result in chunk_results
            if result.requires_llm_fallback
        ]

        return Module3Result(
            chunk_results=chunk_results,
            merged_topics=merged_topics,
            total_chunks=len(chunk_results),
            cs_relevant_chunks=cs_relevant_chunks,
            non_cs_chunks=len(chunk_results) - cs_relevant_chunks,
            official_topic_chunks=classification_counts[
                "official_aqa_topic"
            ],
            mixed_official_unmapped_chunks=classification_counts[
                "mixed_official_and_unmapped"
            ],
            unmapped_cs_chunks=classification_counts[
                "cs_related_unmapped"
            ],
            continuation_chunks=classification_counts[
                "continuation_no_new_topic"
            ],
            no_topic_chunks=classification_counts["no_topic"],
            llm_fallback_chunk_ids=llm_fallback_ids,
            embedding_model=self.extractor.config.embedding_model,
            candidate_keep_threshold=(
                self.relevance_filter.config.candidate_keep_threshold
            ),
        )

    @staticmethod
    def _effective_previous_topic_ids(
        previous_result: ChunkTopicResult | None,
        completed_results: list[ChunkTopicResult],
    ) -> set[str]:
        if previous_result is None:
            return set()

        direct_ids = {
            candidate.concept_id
            for candidate in previous_result.topic_candidates
        }
        if direct_ids:
            return direct_ids

        source_id = previous_result.continuation_of_chunk_id
        if source_id is None:
            return set()

        for result in reversed(completed_results):
            if result.chunk_id != source_id:
                continue

            return {
                candidate.concept_id
                for candidate in result.topic_candidates
            }

        return set()

    def _unmapped_requires_llm_fallback(
        self,
        signals: list[UnmappedCSSignal],
    ) -> bool:
        """
        Escalate only evidence that genuinely needs interpretation.

        Directly accept strong lexical or lexical-semantic rough topics.
        Escalate semantic-only, borderline, or same-evidence ambiguous signals.
        """

        if not signals:
            return False

        for signal in signals:
            if signal.detection_method == "semantic":
                if (
                    signal.score
                    < self.config.strong_unmapped_semantic_score
                ):
                    return True
            elif (
                signal.score
                < self.config.strong_unmapped_lexical_score
            ):
                return True

        evidence_groups: dict[str, list[UnmappedCSSignal]] = {}

        for signal in signals:
            evidence_key = self._normalise_for_decision(signal.evidence)
            evidence_groups.setdefault(evidence_key, []).append(signal)

        for grouped_signals in evidence_groups.values():
            distinct_topics = {
                self._normalise_for_decision(signal.rough_topic)
                for signal in grouped_signals
            }

            if len(distinct_topics) < 2:
                continue

            ordered_scores = sorted(
                (signal.score for signal in grouped_signals),
                reverse=True,
            )

            if (
                ordered_scores[0] - ordered_scores[1]
                <= self.config.ambiguous_signal_margin
            ):
                return True

        return False

    @staticmethod
    def _normalise_for_decision(text: str) -> str:
        text = re.sub(r"[^a-z0-9]+", " ", text.lower())
        return re.sub(r"\s+", " ", text).strip()

    @staticmethod
    def _get_value(item: Any, field_name: str) -> Any:
        if isinstance(item, dict):
            if field_name not in item:
                raise KeyError(f"Missing chunk field: {field_name}")
            return item[field_name]

        if not hasattr(item, field_name):
            raise AttributeError(f"Chunk has no field: {field_name}")
        return getattr(item, field_name)

    @staticmethod
    def _get_optional_value(item: Any, field_name: str) -> Any | None:
        if isinstance(item, dict):
            return item.get(field_name)
        return getattr(item, field_name, None)

# 2. Existing Command-Line Test Scripts

The project test scripts use `argparse` and terminal positional arguments.

Their exact source is retained below as documentation, but they are shown in Markdown rather than executed as complete cells. Executing their `main()` functions directly in Jupyter would cause a `SystemExit` because the notebook kernel does not provide the required command-line arguments.

The interactive execution section uses the same existing helper functions and production classes.

## Existing single-file Module 3 test

**Exact source:** `scripts/test_topic_extraction.py`

```python
from __future__ import annotations

import argparse
import json
import time
from pathlib import Path

from app.services.topic_pipeline import (
    Module3TopicPipeline,
)


PROJECT_ROOT = (
    Path(__file__)
    .resolve()
    .parents[1]
)

TEST_DATA_DIR = (
    PROJECT_ROOT
    / "test_data"
)


def discover_default_input() -> Path:
    candidates = (
        TEST_DATA_DIR
        / "transcript_1_cleaned_chunks.json",
        TEST_DATA_DIR
        / "transcript_1_chunks.json",
    )

    for candidate in candidates:
        if candidate.exists():
            return candidate

    return candidates[0]


def load_chunks(
    input_path: Path,
) -> list[dict]:
    if not input_path.exists():
        raise FileNotFoundError(
            f"Chunk JSON not found: {input_path}\n"
            "Run the Module 2 semantic chunking test first."
        )

    data = json.loads(
        input_path.read_text(
            encoding="utf-8"
        )
    )

    if isinstance(
        data,
        dict,
    ):
        chunks = data.get(
            "chunks"
        )

    elif isinstance(
        data,
        list,
    ):
        chunks = data

    else:
        chunks = None

    if not isinstance(
        chunks,
        list,
    ):
        raise ValueError(
            "Input JSON must be either a chunk list or "
            "an object containing a 'chunks' list."
        )

    return chunks


def model_to_dict(
    model,
) -> dict:
    if hasattr(
        model,
        "model_dump",
    ):
        return model.model_dump()

    return model.dict()


def save_readable_output(
    result,
    output_path: Path,
) -> None:
    lines: list[str] = []

    lines.append(
        "=" * 100
    )

    lines.append(
        "AGENT 1 — MODULE 3 ROUGH TOPIC EXTRACTION"
    )

    lines.append(
        "=" * 100
    )

    lines.append(
        f"Embedding model: "
        f"{result.embedding_model}"
    )

    lines.append(
        f"Candidate keep threshold: "
        f"{result.candidate_keep_threshold}"
    )

    lines.append(
        f"Total chunks: "
        f"{result.total_chunks}"
    )

    lines.append(
        f"CS-relevant chunks: "
        f"{result.cs_relevant_chunks}"
    )

    lines.append(
        f"Non-CS/no-new-topic chunks: "
        f"{result.non_cs_chunks}"
    )

    lines.append(
        f"Official-topic chunks: {result.official_topic_chunks}"
    )
    lines.append(
        "Mixed official + unmapped chunks: "
        f"{result.mixed_official_unmapped_chunks}"
    )
    lines.append(
        f"Unmapped-CS chunks: {result.unmapped_cs_chunks}"
    )
    lines.append(
        f"Continuation/no-new-topic chunks: {result.continuation_chunks}"
    )
    lines.append(
        f"No-topic chunks: {result.no_topic_chunks}"
    )

    lines.append(
        f"LLM fallback chunks: "
        f"{result.llm_fallback_chunk_ids}"
    )

    lines.append("")

    for chunk_result in (
        result.chunk_results
    ):
        lines.append(
            "=" * 100
        )

        lines.append(
            f"CHUNK {chunk_result.chunk_id}"
        )

        lines.append(
            "=" * 100
        )

        lines.append(
            f"Source words: "
            f"{chunk_result.source_word_count}"
        )

        lines.append(
            f"Classification: {chunk_result.classification}"
        )

        lines.append(
            f"CS relevant: "
            f"{chunk_result.is_cs_relevant}"
        )

        lines.append(
            f"Creates new topic: {chunk_result.creates_new_topic}"
        )

        lines.append(
            f"Chunk relevance score: "
            f"{chunk_result.cs_relevance_score}"
        )

        lines.append(
            f"Requires LLM fallback: "
            f"{chunk_result.requires_llm_fallback}"
        )

        if chunk_result.notes:
            lines.append(
                "Notes:"
            )

            for note in (
                chunk_result.notes
            ):
                lines.append(
                    f"- {note}"
                )

        lines.append(
            "\nRETAINED TOPICS"
        )

        lines.append(
            "-" * 100
        )

        if not (
            chunk_result
            .topic_candidates
        ):
            lines.append(
                "None"
            )

        for candidate in (
            chunk_result
            .topic_candidates
        ):
            lines.append(
                f"- {candidate.topic}"
            )

            lines.append(
                f"  concept_id: "
                f"{candidate.concept_id}"
            )

            lines.append(
                f"  domain: "
                f"{candidate.domain}"
            )

            lines.append(
                f"  official reference: {candidate.official_reference}"
            )

            lines.append(
                f"  confidence: "
                f"{candidate.confidence}"
            )

            lines.append(
                f"  salience score: {candidate.salience_score}"
            )

            lines.append(
                f"  keyword score: "
                f"{candidate.keyword_score}"
            )

            lines.append(
                f"  semantic score: "
                f"{candidate.semantic_score}"
            )

            lines.append(
                f"  method: "
                f"{candidate.extraction_method}"
            )

            lines.append(
                f"  matched aliases: "
                f"{candidate.matched_aliases}"
            )

            if candidate.evidence:
                lines.append(
                    "  evidence:"
                )

                for evidence in (
                    candidate.evidence
                ):
                    lines.append(
                        f"    • {evidence}"
                    )

        if chunk_result.has_unmapped_cs_content:
            lines.append("\nUNMAPPED CS SIGNALS")
            lines.append("-" * 100)
            for signal in chunk_result.unmapped_cs_signals:
                lines.append(
                    f"- {signal.rough_topic}"
                )
                lines.append(
                    f"  domain: {signal.domain}"
                )
                lines.append(
                    f"  score: {signal.score}"
                )
                lines.append(
                    f"  method: {signal.detection_method}"
                )
                lines.append(
                    f"  matched aliases: {signal.matched_aliases}"
                )
                lines.append(
                    f"  evidence: {signal.evidence}"
                )

        if (
            chunk_result
            .rejected_candidates
        ):
            lines.append(
                "\nREJECTED / LOW-CONFIDENCE CANDIDATES"
            )

            lines.append(
                "-" * 100
            )

            for candidate in (
                chunk_result
                .rejected_candidates
            ):
                lines.append(
                    f"- {candidate.topic}: "
                    f"{candidate.cs_relevance_score}"
                )

        lines.append("")

    lines.append(
        "=" * 100
    )

    lines.append(
        "MERGED LESSON TOPICS"
    )

    lines.append(
        "=" * 100
    )

    if not result.merged_topics:
        lines.append(
            "No retained CS topics."
        )

    for topic in result.merged_topics:
        lines.append(
            f"- {topic.topic}"
        )

        lines.append(
            f"  concept_id: "
            f"{topic.concept_id}"
        )

        lines.append(
            f"  domain: "
            f"{topic.domain}"
        )

        lines.append(
            f"  role: {topic.topic_role}"
        )

        lines.append(
            f"  confidence: "
            f"{topic.confidence}"
        )

        lines.append(
            f"  ranking score: {topic.ranking_score}"
        )

        lines.append(
            f"  source chunks: "
            f"{topic.source_chunk_ids}"
        )

        lines.append(
            f"  support spans: {topic.support_span_count}"
        )

        lines.append(
            f"  mean semantic score: {topic.mean_semantic_score}"
        )

        lines.append(
            f"  mean salience score: {topic.mean_salience_score}"
        )

        lines.append(
            f"  coverage score: {topic.coverage_score}"
        )

        lines.append(
            f"  supporting candidates: "
            f"{topic.supporting_candidate_count}"
        )

        if topic.evidence:
            lines.append(
                "  evidence:"
            )

            for evidence in (
                topic.evidence
            ):
                lines.append(
                    f"    • {evidence}"
                )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    output_path.write_text(
        "\n".join(lines),
        encoding="utf-8",
    )


def main() -> None:
    parser = argparse.ArgumentParser(
        description=(
            "Test Module 3 rough topic extraction "
            "on Module 2 chunk JSON."
        )
    )

    parser.add_argument(
        "input",
        nargs="?",
        type=Path,
        default=discover_default_input(),
        help=(
            "Path to Module 2 chunk JSON. "
            "Defaults to Transcript 1 chunks."
        ),
    )

    parser.add_argument(
        "--output-json",
        type=Path,
        default=None,
    )

    parser.add_argument(
        "--output-text",
        type=Path,
        default=None,
    )

    args = parser.parse_args()

    chunks = load_chunks(
        args.input
    )

    output_json = (
        args.output_json
        or args.input.with_name(
            args.input.stem
            + "_topics.json"
        )
    )

    output_text = (
        args.output_text
        or args.input.with_name(
            args.input.stem
            + "_topics_readable.txt"
        )
    )

    print(
        "=" * 100
    )

    print(
        "AGENT 1 — MODULE 3 TOPIC EXTRACTION TEST"
    )

    print(
        "=" * 100
    )

    start = time.perf_counter()

    pipeline = Module3TopicPipeline()

    result = pipeline.process_chunks(
        chunks
    )

    elapsed = (
        time.perf_counter()
        - start
    )

    output_json.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    output_json.write_text(
        json.dumps(
            model_to_dict(
                result
            ),
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    save_readable_output(
        result=result,
        output_path=output_text,
    )

    print(
        f"\nProcessing time: "
        f"{elapsed:.3f} seconds"
    )

    print(
        f"Total chunks: "
        f"{result.total_chunks}"
    )

    print(
        f"CS-relevant chunks: "
        f"{result.cs_relevant_chunks}"
    )

    print(
        f"Non-CS chunks: "
        f"{result.non_cs_chunks}"
    )

    print(
        f"Merged topics: "
        f"{len(result.merged_topics)}"
    )

    print(
        f"LLM fallback chunks: "
        f"{result.llm_fallback_chunk_ids}"
    )

    print(
        "\nMERGED TOPICS"
    )

    print(
        "-" * 100
    )

    for topic in result.merged_topics:
        print(
            f"{topic.topic} | "
            f"role={topic.topic_role} | "
            f"ranking={topic.ranking_score} | "
            f"confidence={topic.confidence} | "
            f"chunks={topic.source_chunk_ids}"
        )

    print(
        f"\nJSON saved to:\n"
        f"{output_json}"
    )

    print(
        f"\nReadable output saved to:\n"
        f"{output_text}"
    )

    print(
        "\nMODULE 3 TEST COMPLETED"
    )


if __name__ == "__main__":
    main()
```

The original script remains runnable from the terminal.

## Existing Module 3 batch test

**Exact source:** `scripts/test_module_3_batch.py`

```python
from __future__ import annotations

import argparse
import csv
import json
import time
from collections import Counter
from pathlib import Path
from typing import Any

from app.services.topic_pipeline import Module3TopicPipeline


PROJECT_ROOT = (
    Path(__file__)
    .resolve()
    .parents[1]
)

DEFAULT_INPUT_DIR = (
    PROJECT_ROOT
    / "test_outputs"
    / "module_1_2_batch"
)

DEFAULT_OUTPUT_DIR = (
    PROJECT_ROOT
    / "test_outputs"
    / "module_3_batch"
)


# =========================================================
# GENERIC MODEL SERIALISATION
# =========================================================

def model_to_dict(
    model: Any,
) -> dict[str, Any]:
    """
    Convert a Pydantic model to a plain dictionary.
    """

    if hasattr(
        model,
        "model_dump",
    ):
        return model.model_dump()

    if hasattr(
        model,
        "dict",
    ):
        return model.dict()

    raise TypeError(
        "Expected a Pydantic-style result model."
    )


# =========================================================
# INPUT DISCOVERY
# =========================================================

def is_module_2_chunk_file(
    json_path: Path,
) -> bool:
    """
    Return True only when the JSON file contains Module 2 chunks.

    This avoids processing:
    - batch summaries
    - Module 3 outputs
    - unrelated JSON files
    """

    lower_name = json_path.name.lower()

    excluded_name_parts = (
        "_topics",
        "module_3",
        "batch_summary",
        "summary",
    )

    if any(
        part in lower_name
        for part in excluded_name_parts
    ):
        return False

    try:
        data = json.loads(
            json_path.read_text(
                encoding="utf-8"
            )
        )
    except (
        OSError,
        UnicodeDecodeError,
        json.JSONDecodeError,
    ):
        return False

    if isinstance(
        data,
        dict,
    ):
        chunks = data.get(
            "chunks"
        )

    elif isinstance(
        data,
        list,
    ):
        chunks = data

    else:
        return False

    if not isinstance(
        chunks,
        list,
    ):
        return False

    if not chunks:
        return False

    first_chunk = chunks[0]

    return (
        isinstance(
            first_chunk,
            dict,
        )
        and "chunk_id" in first_chunk
        and "text" in first_chunk
    )


def discover_chunk_files(
    input_dir: Path,
) -> list[Path]:
    """
    Recursively discover all valid Module 2 chunk JSON files.
    """

    if not input_dir.exists():
        raise FileNotFoundError(
            f"Module 1 + 2 batch folder not found: {input_dir}"
        )

    discovered = [
        json_path
        for json_path in input_dir.rglob(
            "*.json"
        )
        if is_module_2_chunk_file(
            json_path
        )
    ]

    return sorted(
        discovered
    )


def load_chunks(
    input_path: Path,
) -> list[dict[str, Any]]:
    """
    Load chunks from one Module 2 JSON file.
    """

    data = json.loads(
        input_path.read_text(
            encoding="utf-8"
        )
    )

    if isinstance(
        data,
        dict,
    ):
        chunks = data[
            "chunks"
        ]
    else:
        chunks = data

    return chunks


# =========================================================
# READABLE OUTPUT
# =========================================================

def save_readable_result(
    result: Any,
    source_file: Path,
    output_path: Path,
) -> None:
    """
    Save a human-readable Module 3 result for one transcript.
    """

    lines: list[str] = []

    lines.append(
        "=" * 110
    )

    lines.append(
        "AGENT 1 — MODULE 3 BATCH TOPIC EXTRACTION"
    )

    lines.append(
        "=" * 110
    )

    lines.append(
        f"Source chunk file: {source_file}"
    )

    lines.append(
        f"Embedding model: {result.embedding_model}"
    )

    lines.append(
        f"Candidate keep threshold: "
        f"{result.candidate_keep_threshold}"
    )

    lines.append(
        f"Total chunks: {result.total_chunks}"
    )

    lines.append(
        f"CS-relevant chunks: "
        f"{result.cs_relevant_chunks}"
    )

    classifications = Counter(
        getattr(
            chunk_result,
            "classification",
            "unknown",
        )
        for chunk_result in result.chunk_results
    )

    lines.append(
        "Classifications: "
        + str(
            dict(
                classifications
            )
        )
    )

    lines.append(
        f"LLM fallback chunks: "
        f"{result.llm_fallback_chunk_ids}"
    )

    lines.append("")

    for chunk_result in result.chunk_results:
        lines.append(
            "=" * 110
        )

        lines.append(
            f"CHUNK {chunk_result.chunk_id}"
        )

        lines.append(
            "=" * 110
        )

        lines.append(
            f"Source words: "
            f"{chunk_result.source_word_count}"
        )

        lines.append(
            f"Classification: "
            f"{chunk_result.classification}"
        )

        lines.append(
            f"CS relevant: "
            f"{chunk_result.is_cs_relevant}"
        )

        lines.append(
            f"Creates new topic: "
            f"{chunk_result.creates_new_topic}"
        )

        lines.append(
            f"Chunk relevance score: "
            f"{chunk_result.cs_relevance_score}"
        )

        lines.append(
            f"Requires LLM fallback: "
            f"{chunk_result.requires_llm_fallback}"
        )

        if chunk_result.notes:
            lines.append(
                "Notes:"
            )

            for note in chunk_result.notes:
                lines.append(
                    f"- {note}"
                )

        lines.append(
            "\nRETAINED OFFICIAL AQA TOPICS"
        )

        lines.append(
            "-" * 110
        )

        if not chunk_result.topic_candidates:
            lines.append(
                "None"
            )

        for candidate in chunk_result.topic_candidates:
            lines.append(
                f"- {candidate.topic}"
            )

            lines.append(
                f"  concept_id: "
                f"{candidate.concept_id}"
            )

            lines.append(
                f"  official reference: "
                f"{getattr(candidate, 'official_reference', None)}"
            )

            lines.append(
                f"  confidence: "
                f"{candidate.confidence}"
            )

            lines.append(
                f"  salience: "
                f"{getattr(candidate, 'salience_score', None)}"
            )

            lines.append(
                f"  keyword score: "
                f"{candidate.keyword_score}"
            )

            lines.append(
                f"  semantic score: "
                f"{candidate.semantic_score}"
            )

            lines.append(
                f"  aliases: "
                f"{candidate.matched_aliases}"
            )

            if candidate.evidence:
                lines.append(
                    "  evidence:"
                )

                for evidence in candidate.evidence:
                    lines.append(
                        f"    • {evidence}"
                    )

        unmapped_signals = getattr(
            chunk_result,
            "unmapped_cs_signals",
            [],
        )

        if unmapped_signals:
            lines.append(
                "\nUNMAPPED CS SIGNALS"
            )

            lines.append(
                "-" * 110
            )

            for signal in unmapped_signals:
                lines.append(
                    f"- {getattr(signal, 'rough_topic', signal.domain)}"
                )

                lines.append(
                    f"  domain: {signal.domain}"
                )

                lines.append(
                    f"  score: {signal.score}"
                )

                lines.append(
                    f"  method: "
                    f"{getattr(signal, 'detection_method', None)}"
                )

                lines.append(
                    f"  aliases: "
                    f"{getattr(signal, 'matched_aliases', [])}"
                )

                lines.append(
                    f"  evidence: {signal.evidence}"
                )

        rejected = getattr(
            chunk_result,
            "rejected_candidates",
            [],
        )

        if rejected:
            lines.append(
                "\nREJECTED / LOW-CONFIDENCE CANDIDATES"
            )

            lines.append(
                "-" * 110
            )

            for candidate in rejected:
                lines.append(
                    f"- {candidate.topic}: "
                    f"{candidate.cs_relevance_score}"
                )

        lines.append("")

    lines.append(
        "=" * 110
    )

    lines.append(
        "MERGED LESSON TOPICS"
    )

    lines.append(
        "=" * 110
    )

    if not result.merged_topics:
        lines.append(
            "No merged official AQA topics."
        )

    for topic in result.merged_topics:
        lines.append(
            f"- {topic.topic}"
        )

        lines.append(
            f"  concept_id: {topic.concept_id}"
        )

        lines.append(
            f"  role: "
            f"{getattr(topic, 'topic_role', None)}"
        )

        lines.append(
            f"  confidence: {topic.confidence}"
        )

        lines.append(
            f"  ranking score: "
            f"{getattr(topic, 'ranking_score', None)}"
        )

        lines.append(
            f"  source chunks: "
            f"{topic.source_chunk_ids}"
        )

        lines.append(
            f"  support spans: "
            f"{getattr(topic, 'support_span_count', None)}"
        )

        lines.append(
            f"  mean semantic score: "
            f"{getattr(topic, 'mean_semantic_score', None)}"
        )

        lines.append(
            f"  mean salience score: "
            f"{getattr(topic, 'mean_salience_score', None)}"
        )

        lines.append(
            f"  coverage score: "
            f"{getattr(topic, 'coverage_score', None)}"
        )

        if topic.evidence:
            lines.append(
                "  evidence:"
            )

            for evidence in topic.evidence:
                lines.append(
                    f"    • {evidence}"
                )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    output_path.write_text(
        "\n".join(
            lines
        ),
        encoding="utf-8",
    )


# =========================================================
# SUMMARY CREATION
# =========================================================

def build_summary_row(
    source_file: Path,
    result: Any,
    processing_time: float,
) -> dict[str, Any]:
    classifications = Counter(
        getattr(
            chunk_result,
            "classification",
            "unknown",
        )
        for chunk_result in result.chunk_results
    )

    merged_primary = [
        topic.topic
        for topic in result.merged_topics
        if getattr(
            topic,
            "topic_role",
            None,
        ) == "primary"
    ]

    merged_supporting = [
        topic.topic
        for topic in result.merged_topics
        if getattr(
            topic,
            "topic_role",
            None,
        ) == "supporting"
    ]

    tracing_chunks = [
        chunk_result.chunk_id
        for chunk_result in result.chunk_results
        if any(
            candidate.concept_id
            == "aqa_3_1_1_algorithm_purpose_trace"
            for candidate in (
                chunk_result.topic_candidates
            )
        )
    ]

    unmapped_signal_count = sum(
        len(
            getattr(
                chunk_result,
                "unmapped_cs_signals",
                [],
            )
        )
        for chunk_result in result.chunk_results
    )

    return {
        "source_file": str(
            source_file
        ),
        "processing_time_seconds": round(
            processing_time,
            4,
        ),
        "total_chunks": result.total_chunks,
        "cs_relevant_chunks": (
            result.cs_relevant_chunks
        ),
        "official_topic_chunks": (
            classifications[
                "official_aqa_topic"
            ]
        ),
        "mixed_official_unmapped_chunks": (
            classifications[
                "mixed_official_and_unmapped"
            ]
        ),
        "unmapped_cs_chunks": (
            classifications[
                "cs_related_unmapped"
            ]
        ),
        "continuation_chunks": (
            classifications[
                "continuation_no_new_topic"
            ]
        ),
        "no_topic_chunks": (
            classifications[
                "no_topic"
            ]
        ),
        "llm_fallback_chunks": (
            ",".join(
                str(chunk_id)
                for chunk_id in (
                    result
                    .llm_fallback_chunk_ids
                )
            )
        ),
        "tracing_chunks": ",".join(
            str(chunk_id)
            for chunk_id in tracing_chunks
        ),
        "unmapped_signal_count": (
            unmapped_signal_count
        ),
        "merged_topic_count": len(
            result.merged_topics
        ),
        "primary_topics": " | ".join(
            merged_primary
        ),
        "supporting_topics": " | ".join(
            merged_supporting
        ),
    }


def save_summary_csv(
    rows: list[dict[str, Any]],
    output_path: Path,
) -> None:
    if not rows:
        return

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with output_path.open(
        "w",
        encoding="utf-8-sig",
        newline="",
    ) as csv_file:
        writer = csv.DictWriter(
            csv_file,
            fieldnames=list(
                rows[0].keys()
            ),
        )

        writer.writeheader()
        writer.writerows(
            rows
        )


# =========================================================
# MAIN
# =========================================================

def main() -> None:
    parser = argparse.ArgumentParser(
        description=(
            "Run Agent 1 Module 3 over every Module 2 "
            "chunk JSON in the batch output folder."
        )
    )

    parser.add_argument(
        "--input-dir",
        type=Path,
        default=DEFAULT_INPUT_DIR,
    )

    parser.add_argument(
        "--output-dir",
        type=Path,
        default=DEFAULT_OUTPUT_DIR,
    )

    parser.add_argument(
        "--limit",
        type=int,
        default=None,
        help=(
            "Optionally test only the first N discovered transcripts."
        ),
    )

    args = parser.parse_args()

    chunk_files = discover_chunk_files(
        args.input_dir
    )

    if args.limit is not None:
        chunk_files = chunk_files[
            : args.limit
        ]

    if not chunk_files:
        raise RuntimeError(
            "No Module 2 chunk JSON files were discovered in "
            f"{args.input_dir}"
        )

    print(
        "=" * 110
    )

    print(
        "AGENT 1 — MODULE 3 BATCH EVALUATION"
    )

    print(
        "=" * 110
    )

    print(
        f"\nChunk files found: {len(chunk_files)}"
    )

    for index, chunk_file in enumerate(
        chunk_files,
        start=1,
    ):
        print(
            f"{index}. "
            f"{chunk_file.relative_to(args.input_dir)}"
        )

    pipeline = Module3TopicPipeline()

    summary_rows: list[
        dict[str, Any]
    ] = []

    for index, chunk_file in enumerate(
        chunk_files,
        start=1,
    ):
        print(
            "\n"
            + "=" * 110
        )

        print(
            f"TESTING {index}/{len(chunk_files)}: "
            f"{chunk_file.name}"
        )

        print(
            "=" * 110
        )

        chunks = load_chunks(
            chunk_file
        )

        started = time.perf_counter()

        result = pipeline.process_chunks(
            chunks
        )

        elapsed = (
            time.perf_counter()
            - started
        )

        relative_parent = (
            chunk_file
            .parent
            .relative_to(
                args.input_dir
            )
        )

        transcript_output_dir = (
            args.output_dir
            / relative_parent
            / chunk_file.stem
        )

        json_output = (
            transcript_output_dir
            / "module_3_topics.json"
        )

        readable_output = (
            transcript_output_dir
            / "module_3_topics_readable.txt"
        )

        json_output.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        json_output.write_text(
            json.dumps(
                model_to_dict(
                    result
                ),
                indent=2,
                ensure_ascii=False,
            ),
            encoding="utf-8",
        )

        save_readable_result(
            result=result,
            source_file=chunk_file,
            output_path=readable_output,
        )

        summary_row = build_summary_row(
            source_file=chunk_file,
            result=result,
            processing_time=elapsed,
        )

        summary_rows.append(
            summary_row
        )

        print(
            f"Processing time: {elapsed:.3f}s"
        )

        print(
            f"Total chunks: {result.total_chunks}"
        )

        print(
            f"CS-relevant chunks: "
            f"{result.cs_relevant_chunks}"
        )

        print(
            f"Merged topics: "
            f"{len(result.merged_topics)}"
        )

        print(
            f"Tracing chunks: "
            f"{summary_row['tracing_chunks'] or 'None'}"
        )

        print(
            f"LLM fallback chunks: "
            f"{summary_row['llm_fallback_chunks'] or 'None'}"
        )

        print(
            "Primary topics: "
            f"{summary_row['primary_topics'] or 'None'}"
        )

        print(
            "Supporting topics: "
            f"{summary_row['supporting_topics'] or 'None'}"
        )

    summary_json = (
        args.output_dir
        / "module_3_batch_summary.json"
    )

    summary_csv = (
        args.output_dir
        / "module_3_batch_summary.csv"
    )

    args.output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    summary_json.write_text(
        json.dumps(
            summary_rows,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    save_summary_csv(
        rows=summary_rows,
        output_path=summary_csv,
    )

    print(
        "\n"
        + "=" * 110
    )

    print(
        "MODULE 3 BATCH TEST COMPLETED"
    )

    print(
        "=" * 110
    )

    print(
        f"\nDetailed outputs:\n"
        f"{args.output_dir}"
    )

    print(
        f"\nSummary JSON:\n"
        f"{summary_json}"
    )

    print(
        f"\nSummary CSV:\n"
        f"{summary_csv}"
    )


if __name__ == "__main__":
    main()
```

The original script remains runnable from the terminal.

# 3. Import the Existing Production Classes and Test Helpers

No Module 3 algorithm is recreated in the notebook.

The notebook imports:

- `Module3TopicPipeline` from the existing application;
- the exact input-discovery, loading, readable-output and summary helpers from the existing batch test script.

In [ ]:
import csv
import json
import time
from collections import Counter
from typing import Any

import pandas as pd

from app.services.topic_pipeline import (
    Module3TopicPipeline,
)
from scripts.test_module_3_batch import (
    build_summary_row,
    discover_chunk_files,
    is_module_2_chunk_file,
    load_chunks,
    model_to_dict,
    save_readable_result,
    save_summary_csv,
)

print("Existing Module 3 pipeline imported successfully.")
print("Existing Module 3 batch helpers imported successfully.")

# 4. Discover Real Module 2 Chunk Files

The notebook searches only for real Module 2 JSON outputs.

It checks:

```text
test_outputs/module_1_2_batch/
test_outputs/pipeline_runs/
test_data/
```

The existing `is_module_2_chunk_file()` helper rejects:

- Module 3 topic outputs;
- batch summaries;
- unrelated JSON files;
- JSON files without a valid `chunks` list.

No fake chunks are created.

In [ ]:
SEARCH_LOCATIONS = [
    (
        PROJECT_ROOT
        / "test_outputs"
        / "module_1_2_batch"
    ),
    (
        PROJECT_ROOT
        / "test_outputs"
        / "pipeline_runs"
    ),
    (
        PROJECT_ROOT
        / "test_data"
    ),
]

discovered_chunk_files: set[Path] = set()

for location in SEARCH_LOCATIONS:
    if not location.exists():
        continue

    for json_path in location.rglob(
        "*.json"
    ):
        if is_module_2_chunk_file(
            json_path
        ):
            discovered_chunk_files.add(
                json_path.resolve()
            )

chunk_files = sorted(
    discovered_chunk_files,
    key=lambda path: str(path).casefold(),
)

if not chunk_files:
    raise FileNotFoundError(
        "No real Module 2 chunk JSON files were found."
    )

print(
    "Real Module 2 chunk files discovered:",
    len(chunk_files),
)

for index, path in enumerate(
    chunk_files,
    start=1,
):
    print(
        f"{index:02d}. "
        f"{path.relative_to(PROJECT_ROOT)}"
    )

# 5. Select One Real Chunk File

The first run uses one discovered Module 2 output.

Change the index only when another real transcript should be inspected. The Module 3 logic remains unchanged.

In [ ]:
PRIMARY_CHUNK_PATH = chunk_files[0]

primary_chunks = load_chunks(
    PRIMARY_CHUNK_PATH
)

print(
    "Selected chunk file:",
    PRIMARY_CHUNK_PATH.relative_to(
        PROJECT_ROOT
    ),
)

print(
    "Chunk count:",
    len(primary_chunks),
)

print(
    "\nFirst chunk fields:",
    sorted(
        primary_chunks[0].keys()
    ),
)

print(
    "\nFirst chunk preview:\n"
)

print(
    str(
        primary_chunks[0].get(
            "text",
            "",
        )
    )[:1500]
)

# 6. Run the Exact Existing Module 3 Pipeline

This cell instantiates `Module3TopicPipeline()` with its existing default components and configuration.

The pipeline itself calculates:

- official topic candidates;
- keyword scores;
- semantic scores;
- salience scores;
- confidence;
- CS relevance;
- chunk classifications;
- continuation handling;
- unmapped-CS signals;
- LLM-fallback flags;
- merged topic confidence;
- topic roles;
- final ranking.

Nothing is supplied manually apart from the real Module 2 chunks.

In [ ]:
topic_pipeline = Module3TopicPipeline()

single_start = time.perf_counter()

primary_result = (
    topic_pipeline.process_chunks(
        primary_chunks
    )
)

single_runtime = (
    time.perf_counter()
    - single_start
)

print(
    json.dumps(
        model_to_dict(
            primary_result
        ),
        indent=2,
        ensure_ascii=False,
    )
)

print(
    f"\nProcessing time: "
    f"{single_runtime:.4f} seconds"
)

# 7. Display the Real Merged Topics

This is a presentation view of the actual `merged_topics` returned by the existing pipeline.

The table does not alter ranking or confidence.

In [ ]:
merged_topic_rows = [
    {
        "Rank": rank,
        "Topic": topic.topic,
        "Concept ID": topic.concept_id,
        "Domain": topic.domain,
        "Official reference": (
            topic.official_reference
        ),
        "Chapter reference": (
            topic.chapter_reference
        ),
        "Paper": topic.paper,
        "Role": topic.topic_role,
        "Confidence": topic.confidence,
        "Ranking score": (
            topic.ranking_score
        ),
        "Mean semantic score": (
            topic.mean_semantic_score
        ),
        "Mean keyword score": (
            topic.mean_keyword_score
        ),
        "Mean salience score": (
            topic.mean_salience_score
        ),
        "Coverage score": (
            topic.coverage_score
        ),
        "Source chunks": (
            topic.source_chunk_ids
        ),
        "Support spans": (
            topic.support_span_count
        ),
    }
    for rank, topic in enumerate(
        primary_result.merged_topics,
        start=1,
    )
]

merged_topics_dataframe = pd.DataFrame(
    merged_topic_rows
)

merged_topics_dataframe

# 8. Display Real Chunk-by-Chunk Classifications

Each row comes directly from one `ChunkTopicResult`.

This view shows whether a real chunk was classified as:

- `official_aqa_topic`;
- `mixed_official_and_unmapped`;
- `cs_related_unmapped`;
- `continuation_no_new_topic`;
- `no_topic`.

In [ ]:
chunk_result_rows = [
    {
        "Chunk ID": chunk_result.chunk_id,
        "Source words": (
            chunk_result.source_word_count
        ),
        "Classification": (
            chunk_result.classification
        ),
        "CS relevant": (
            chunk_result.is_cs_relevant
        ),
        "Creates new topic": (
            chunk_result.creates_new_topic
        ),
        "CS relevance score": (
            chunk_result.cs_relevance_score
        ),
        "Retained topics": [
            candidate.topic
            for candidate
            in chunk_result.topic_candidates
        ],
        "Rejected topics": [
            candidate.topic
            for candidate
            in chunk_result.rejected_candidates
        ],
        "Has unmapped CS": (
            chunk_result
            .has_unmapped_cs_content
        ),
        "Unmapped signals": [
            signal.rough_topic
            for signal
            in chunk_result.unmapped_cs_signals
        ],
        "Continuation of": (
            chunk_result
            .continuation_of_chunk_id
        ),
        "Requires LLM fallback": (
            chunk_result
            .requires_llm_fallback
        ),
    }
    for chunk_result
    in primary_result.chunk_results
]

chunk_results_dataframe = pd.DataFrame(
    chunk_result_rows
)

chunk_results_dataframe

# 9. Save the Real Single-Transcript Result

The notebook uses the existing readable-output helper from `scripts/test_module_3_batch.py`.

The output includes:

- complete JSON;
- human-readable topic report;
- source chunk reference;
- classifications;
- retained and rejected candidates;
- merged topics;
- fallback flags.

In [ ]:
SINGLE_OUTPUT_ROOT = (
    PROJECT_ROOT
    / "test_outputs"
    / "module_3_exact_notebook"
    / "single"
)

SINGLE_OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

single_json_path = (
    SINGLE_OUTPUT_ROOT
    / "topics.json"
)

single_readable_path = (
    SINGLE_OUTPUT_ROOT
    / "topics_readable.txt"
)

single_json_path.write_text(
    json.dumps(
        model_to_dict(
            primary_result
        ),
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

save_readable_result(
    result=primary_result,
    source_file=PRIMARY_CHUNK_PATH,
    output_path=single_readable_path,
)

print(f"JSON: {single_json_path}")
print(
    f"Readable output: "
    f"{single_readable_path}"
)

## Existing batch-helper signature

The current project helper is used with its exact signature:

```python
build_summary_row(
    source_file=chunk_path,
    result=result,
    processing_time=elapsed,
)
```

The summary field produced by the helper is:

```text
processing_time_seconds
```

This notebook does not rename or replace the existing helper.

# 10. Run the Exact Pipeline on the Complete Real Batch

Every discovered Module 2 chunk file is processed independently.

A failure in one file is recorded without stopping the remaining batch.

For each transcript, the notebook saves:

```text
topics.json
topics_readable.txt
```

and creates the same summary structure used by the existing Module 3 batch test.

In [ ]:
BATCH_OUTPUT_ROOT = (
    PROJECT_ROOT
    / "test_outputs"
    / "module_3_exact_notebook"
    / "batch"
)

BATCH_OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

batch_pipeline = Module3TopicPipeline()

batch_rows: list[dict[str, Any]] = []
batch_errors: list[dict[str, Any]] = []

batch_start = time.perf_counter()

for index, chunk_path in enumerate(
    chunk_files,
    start=1,
):
    relative_path = (
        chunk_path.relative_to(
            PROJECT_ROOT
        )
    )

    print()
    print("=" * 110)
    print(
        f"[{index}/{len(chunk_files)}] "
        f"{relative_path}"
    )
    print("=" * 110)

    try:
        chunks = load_chunks(
            chunk_path
        )

        start = time.perf_counter()

        result = (
            batch_pipeline.process_chunks(
                chunks
            )
        )

        elapsed = (
            time.perf_counter()
            - start
        )

        output_name = (
            chunk_path.parent.name
            + "_"
            + chunk_path.stem
        )

        output_name = re.sub(
            r"[^A-Za-z0-9_-]+",
            "_",
            output_name,
        ).strip("_")

        transcript_output_dir = (
            BATCH_OUTPUT_ROOT
            / output_name
        )

        transcript_output_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        topics_json_path = (
            transcript_output_dir
            / "topics.json"
        )

        readable_path = (
            transcript_output_dir
            / "topics_readable.txt"
        )

        topics_json_path.write_text(
            json.dumps(
                model_to_dict(
                    result
                ),
                indent=2,
                ensure_ascii=False,
            ),
            encoding="utf-8",
        )

        save_readable_result(
            result=result,
            source_file=chunk_path,
            output_path=readable_path,
        )

        summary_row = build_summary_row(
            result=result,
            source_file=chunk_path,
            processing_time=elapsed,
        )

        batch_rows.append(
            summary_row
        )

        print(
            f"Completed in "
            f"{elapsed:.4f} seconds"
        )

        print(
            f"Chunks: {result.total_chunks}"
        )

        print(
            "CS-relevant chunks: "
            f"{result.cs_relevant_chunks}"
        )

        print(
            "Merged topics: "
            f"{len(result.merged_topics)}"
        )

        print(
            "Classifications: "
            f"{dict(Counter(
                item.classification
                for item
                in result.chunk_results
            ))}"
        )

        print(
            "LLM fallback chunks: "
            f"{result.llm_fallback_chunk_ids}"
        )

        print(
            "Merged topic names: "
            f"{[
                topic.topic
                for topic
                in result.merged_topics
            ]}"
        )

    except Exception as exc:
        error_row = {
            "source_file": str(
                relative_path
            ),
            "error_type": (
                type(exc).__name__
            ),
            "error_message": str(exc),
        }

        batch_errors.append(
            error_row
        )

        print(
            f"FAILED: "
            f"{type(exc).__name__}: {exc}"
        )

batch_runtime = (
    time.perf_counter()
    - batch_start
)

print()
print("=" * 110)
print("MODULE 3 REAL BATCH COMPLETED")
print("=" * 110)
print(
    f"Successful files: "
    f"{len(batch_rows)}"
)
print(
    f"Failed files: "
    f"{len(batch_errors)}"
)
print(
    f"Total runtime: "
    f"{batch_runtime:.2f} seconds"
)

# 11. Save the Existing Batch Summary Format

The exact existing `save_summary_csv()` helper is used.

The notebook also saves failures separately so no missing file is hidden.

In [ ]:
summary_json_path = (
    BATCH_OUTPUT_ROOT
    / "module_3_batch_summary.json"
)

summary_csv_path = (
    BATCH_OUTPUT_ROOT
    / "module_3_batch_summary.csv"
)

errors_json_path = (
    BATCH_OUTPUT_ROOT
    / "module_3_batch_errors.json"
)

summary_json_path.write_text(
    json.dumps(
        batch_rows,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

save_summary_csv(
    rows=batch_rows,
    output_path=summary_csv_path,
)

errors_json_path.write_text(
    json.dumps(
        batch_errors,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print(f"Summary JSON: {summary_json_path}")
print(f"Summary CSV: {summary_csv_path}")
print(f"Errors JSON: {errors_json_path}")

# 12. Display the Real Batch Summary

This table contains the exact summary rows created by the existing `build_summary_row()` helper.

In [ ]:
batch_summary_dataframe = pd.DataFrame(
    batch_rows
)

batch_summary_dataframe

# 13. Aggregate the Real Module 3 Results

The following analysis is derived only from the real batch summary.

It does not define expected topics or alter any pipeline output.

In [ ]:
aggregate_summary = {
    "successful_transcripts": len(
        batch_rows
    ),
    "failed_transcripts": len(
        batch_errors
    ),
    "total_source_chunks": sum(
        int(
            row.get(
                "total_chunks",
                0,
            )
        )
        for row in batch_rows
    ),
    "total_cs_relevant_chunks": sum(
        int(
            row.get(
                "cs_relevant_chunks",
                0,
            )
        )
        for row in batch_rows
    ),
    "total_official_topic_chunks": sum(
        int(
            row.get(
                "official_topic_chunks",
                0,
            )
        )
        for row in batch_rows
    ),
    "total_mixed_official_unmapped_chunks": sum(
        int(
            row.get(
                "mixed_official_unmapped_chunks",
                0,
            )
        )
        for row in batch_rows
    ),
    "total_unmapped_cs_chunks": sum(
        int(
            row.get(
                "unmapped_cs_chunks",
                0,
            )
        )
        for row in batch_rows
    ),
    "total_continuation_chunks": sum(
        int(
            row.get(
                "continuation_chunks",
                0,
            )
        )
        for row in batch_rows
    ),
    "total_no_topic_chunks": sum(
        int(
            row.get(
                "no_topic_chunks",
                0,
            )
        )
        for row in batch_rows
    ),
    "total_unmapped_signals": sum(
        int(
            row.get(
                "unmapped_signal_count",
                0,
            )
        )
        for row in batch_rows
    ),
    "total_merged_topics": sum(
        int(
            row.get(
                "merged_topic_count",
                0,
            )
        )
        for row in batch_rows
    ),
    "average_processing_time_seconds": (
        round(
            sum(
                float(
                    row.get(
                        "processing_time_seconds",
                        0.0,
                    )
                )
                for row in batch_rows
            )
            / len(batch_rows),
            4,
        )
        if batch_rows
        else 0.0
    ),
}

print(
    json.dumps(
        aggregate_summary,
        indent=2,
        ensure_ascii=False,
    )
)

# 14. Module 3 Evaluation Guidance

The real outputs should be reviewed using the following factors:

## Candidate extraction

- Were clearly discussed official AQA topics retained?
- Were incidental ordinary words prevented from becoming topics?
- Are semantic-only candidates supported by appropriate context?

## Chunk classification

- Are non-CS chunks classified as `no_topic`?
- Are overlap-only chunks treated as continuations rather than new topics?
- Are mixed official and unmapped discussions represented correctly?

## Topic merging

- Are repeated concepts merged across chunks?
- Do adjacent size-split chunks remain one support span?
- Are primary and supporting roles plausible?
- Does ranking favour strongly explained lesson topics?

## False positives and omissions

The pipeline output should be compared with the actual transcript content. Any observed false positive or missing topic should be recorded as evaluation evidence rather than corrected manually inside the notebook.

## LLM fallback

`requires_llm_fallback` is only a decision flag in Module 3. The current pipeline does not invent a syllabus label for uncertain or unmapped content.

# 15. Final Module 3 Decision

## Existing logic

This notebook retains the exact current Module 3 architecture:

```text
Module 2 chunks
    ↓
Official AQA catalogue retrieval
    ↓
Keyword and MiniLM semantic candidate evidence
    ↓
Salience-aware candidate scoring
    ↓
Relevance filtering
    ↓
Continuation handling
    ↓
Unmapped-CS detection
    ↓
Fallback decision
    ↓
Topic merging, role assignment and ranking
```

No implementation rule was changed during notebook migration.

## Final evaluation status

The winner or final acceptance of Module 3 should be based on the real outputs shown above:

- correct official topics;
- false positives;
- missed topics;
- classification quality;
- continuation handling;
- merged-topic ranking;
- fallback behaviour.

Any refinements identified from these outputs should first be made in the production `.py` files and then remigrated into the notebook, preserving one source of truth.

# 16. Presentation Checklist

Show:

1. The exact existing Module 3 schemas.
2. The exact AQA concept catalogue.
3. The exact candidate extractor.
4. The exact relevance filter.
5. The exact unmapped-CS detector.
6. The exact topic merger.
7. The exact Module 3 pipeline.
8. The real Module 2 chunk file selected.
9. The full real JSON result.
10. The merged-topic table.
11. The chunk-by-chunk classification table.
12. The complete real batch results.
13. Saved JSON, readable text and CSV outputs.
14. Observed false positives and missed topics.
15. The final decision based on real pipeline evidence.

No expected topic list or confidence score is hardcoded in this notebook.